# arc3-flashnext-smoke — STOCK duck × Flash-Next NVFP4 (the model-swap read)

One phase, all 25 public games, eval geometry (7,920 s/game, concurrency 28
from the serialized solver), served by sonpham's **Qwen3.8-Flash-Next NVFP4**
package on ONE RTX Pro 6000 (gate-proven boot: 740 s on rung `gcp_exact`,
`submission/_flashnext_gate/results_v4/`). The harness is the **stock duck**
(anim-20260807 bundle): no grafts, `ONLY_RESET_LEVELS=true`, stock sampling
(temp 0.6 / top-p 0.95 / top-k 20).

**Deviation vs the 27B baseline:** `LOCAL_ANALYZER_CONTEXT_WINDOW=24576` +
`LOCAL_ANALYZER_MAX_OUTPUT=4096`, because their server serves
`--max-model-len 32768` (the 27B ran a 65536-ctx server with a 32768 window
and no max_tokens). Fair enough: the stock 27B's effective history is 4-9
turns anyway.

**Pre-registered read** — baseline = pooled stock 27B (3 kernels: levels/game
1.00 / 1.04 / 0.84-0.88, zero-level 8-9/25, mean local score 3.5-4.9):

* **PASS** = flashnext levels/game >= 1.3 OR zero-level <= 6 with levels >= 1.0
* **INCONCLUSIVE** = levels/game 0.9-1.3
* **FAIL** = levels/game < 0.9

Also record per-game actions (expect MORE actions/game from the faster
serving: gate conc-28 measured 412 vs the 27B's 297 gen tok/s aggregate) and
the server queue (28 workers vs `--max-num-seqs 22` — requests queue by
design; running/waiting/kv sampled every 60 s).


In [ ]:
import contextlib
import json
import os
import pickle
import subprocess
import sys
import time
from datetime import datetime, timedelta
from pathlib import Path
from typing import TextIO
from urllib.request import urlopen


def _env_bool(name: str, default: bool = False) -> bool:
    raw = os.getenv(name, "").strip().lower()
    if not raw:
        return default
    return raw in {"1", "true", "yes", "y", "on"}


NOTEBOOK_START_EPOCH = time.time()
RUN_AS_SUBMISSION = False
RUN_AS_SUBMISSION = RUN_AS_SUBMISSION or _env_bool("KAGGLE_IS_COMPETITION_RERUN", False)
ENABLE_GPU = True

os.environ["TAAF_RUN_AS_SUBMISSION"] = "1" if RUN_AS_SUBMISSION else "0"
os.environ.setdefault("MPLBACKEND", "Agg")

if ENABLE_GPU:
    cuda_library_path = "/usr/local/nvidia/lib64"
    existing = [entry for entry in os.environ.get("LIBRARY_PATH", "").split(os.pathsep) if entry]
    os.environ["LIBRARY_PATH"] = os.pathsep.join(
        [cuda_library_path, *[entry for entry in existing if entry != cuda_library_path]]
    )

print(f"TAAF RUN_AS_SUBMISSION={RUN_AS_SUBMISSION}")
if ENABLE_GPU:
    print(f"taaf.kaggle: LIBRARY_PATH={os.environ['LIBRARY_PATH']}")

In [ ]:
# Fail-fast GPU assert: metadata machine_shape + --accelerator alone can still
# bind P100; the competition source attachment is the real RTX Pro 6000 gate.
import subprocess as _sp

_gpu = _sp.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
    capture_output=True, text=True,
)
print("boot gpu:", (_gpu.stdout or "").strip() or (_gpu.stderr or "").strip())
_gpu_name = (_gpu.stdout or "").upper()
assert "RTX" in _gpu_name and "6000" in _gpu_name, (
    f"GPU misbind — expected RTX Pro 6000, got: {_gpu.stdout!r} {_gpu.stderr!r}"
)


In [ ]:
wheelhouse = Path("/kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels")
if wheelhouse.exists():
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--no-index",
            "--no-warn-conflicts",
            "--disable-pip-version-check",
            "--find-links",
            str(wheelhouse),
            "arc-agi",
        ]
    )
elif os.getenv("TAAF_KAGGLE_BUNDLE_DIR"):
    print(f"Competition wheelhouse not found at {wheelhouse}; assuming local debug dependencies are installed.")
else:
    raise RuntimeError(f"Competition wheelhouse not found at {wheelhouse}.")

In [ ]:
# ================= flashnext-smoke config — bundle, sys.path, results sink =================
# REPLACES the v12 "Qwen3.8 / Kaggle input configuration" cell. The 27B Kaggle
# Model is deliberately NOT attached: the model axis IS the experiment. The
# serving stack (sonpham's Flash-Next NVFP4 package) is assembled below.
import re
import shutil
import signal
import traceback
import urllib.request

WORKING_DIR = Path(os.getenv("TAAF_KAGGLE_WORKING_DIR", "/kaggle/working")).resolve()
WORKING_DIR.mkdir(parents=True, exist_ok=True)
SETUP_ENV_PATH = WORKING_DIR / "taaf_setup_env.json"
SOFT_DEADLINE_BUFFER_S = 600.0
DATASET_BUNDLE_MARKER = "taaf-kaggle-bundle.json"

# Keep the whole run offline. vLLM/Transformers must use the mounted files only.
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["TAAF_KAGGLE_WORKING_DIR"] = str(WORKING_DIR)
os.environ["TAAF_KAGGLE_SETUP_ENV"] = str(SETUP_ENV_PATH)


def _find_taaf_bundle() -> Path:
    explicit = os.getenv("TAAF_KAGGLE_BUNDLE_DIR", "").strip()
    # v2: sonpham's flashnext datasets ALSO contain a taaf-kaggle-bundle.json
    # (their apex fork). v1 rglobbed /kaggle/input and picked theirs first ->
    # their solver source + our pickle -> AttributeError hard_noop_guard.
    # Pin to the anim-20260807 bundle (the bytes the pickle was built from).
    for pinned in (Path("/kaggle/input/taaf-kaggle-source-anim-20260807-anim"),
                   Path("/kaggle/input/datasets/jakobbrggen/taaf-kaggle-source-anim-20260807-anim")):
        if (pinned / DATASET_BUNDLE_MARKER).is_file():
            return pinned
    if explicit and (Path(explicit) / DATASET_BUNDLE_MARKER).is_file():
        return Path(explicit)
    for root in [Path("/kaggle/input/datasets"), Path("/kaggle/input"), Path.cwd()]:
        if root.exists():
            for marker in root.rglob(DATASET_BUNDLE_MARKER):
                return marker.parent
    raise RuntimeError("Could not find TAAF Kaggle source bundle dataset.")


BUNDLE_DIR = _find_taaf_bundle()
TAAF_BUNDLE_DIR = BUNDLE_DIR    # survives the gate assemble cell, which REBINDS BUNDLE_DIR
os.environ["TAAF_KAGGLE_BUNDLE_DIR"] = str(BUNDLE_DIR)
print(f"TAAF source bundle: {BUNDLE_DIR}")


# Make bundled TAAF repos importable for this notebook and child Python
# processes (verbatim from the v12 setup cell — the part we keep).
def _source_path_entries(bundle_dir: Path) -> list[Path]:
    src_root = bundle_dir / "src"
    if not src_root.is_dir():
        return []
    entries: list[Path] = []
    for repo in sorted(src_root.iterdir(), reverse=True):
        if not repo.is_dir():
            continue
        for candidate in (repo / "src", repo):
            if candidate.is_dir():
                entries.append(candidate)
    return entries


source_entries = _source_path_entries(BUNDLE_DIR)
for entry in source_entries:
    if str(entry) not in sys.path:
        sys.path.insert(0, str(entry))
if source_entries:
    import sysconfig

    pth_path = Path(sysconfig.get_paths()["purelib"]) / "taaf_kaggle_sources.pth"
    pth_path.write_text("".join(f"{entry}\n" for entry in source_entries), encoding="utf-8")
    print(f"taaf.kaggle: wrote {pth_path} ({len(source_entries)} source roots)", flush=True)

# ---- boot-results sink (flashnext-gate pattern): partial state always lands on disk ----
RESULTS_PATH = WORKING_DIR / "flashnext_smoke_boot.json"
RESULTS = {
    "meta": {"kernel": "arc3-flashnext-smoke",
             "started_utc": datetime.utcnow().isoformat() + "Z"},
    "boots": {},
    "phases": {},
    "verdicts": {},
}


def elapsed_min():
    return (time.time() - NOTEBOOK_START_EPOCH) / 60.0


def save_results():
    tmp = RESULTS_PATH.with_suffix(".tmp")
    tmp.write_text(json.dumps(RESULTS, indent=2, default=str) + "\n", encoding="utf-8")
    tmp.replace(RESULTS_PATH)


def host_snapshot(tag):
    snap = {}
    try:
        mem = dict((ln.split(":", 1)[0], ln.split(":", 1)[1].strip())
                   for ln in Path("/proc/meminfo").read_text().splitlines() if ":" in ln)
        snap["mem_total"] = mem.get("MemTotal")
        snap["mem_available"] = mem.get("MemAvailable")
    except Exception as exc:
        snap["meminfo_error"] = repr(exc)[:120]
    for mount in ("/kaggle/working", "/kaggle/tmp", "/tmp", "/"):
        try:
            usage = shutil.disk_usage(mount)
            snap[mount] = f"free {usage.free / 1e9:.1f} / total {usage.total / 1e9:.1f} GB"
        except Exception:
            snap[mount] = "n/a"
    RESULTS["meta"].setdefault("host", {})[tag] = snap
    return snap


print("flashnext-smoke: host snapshot", json.dumps(host_snapshot("start"), indent=1), flush=True)
save_results()


In [ ]:
# Audit the attached inputs that matter for this run. NOTE: no 27B model
# mount — the Kaggle models input dir is expected to be ABSENT.
INPUT_ROOT = Path("/kaggle/input")
print("=== TAAF bundle ===")
print(BUNDLE_DIR, "exists:", BUNDLE_DIR.exists())
for _needle in ("serving-part-000", "serving-part-001", "serving-part-002"):
    _hits = sorted({str(p) for p in INPUT_ROOT.rglob(_needle) if p.is_dir()})
    print(f"=== {_needle} ===", _hits)
_tarballs = sorted({str(p) for p in INPUT_ROOT.rglob("flashnext-gcp-container-site-packages.tar.zst")})
print("=== runtime tarball ===", _tarballs)
_wheels = [str(p) for p in (INPUT_ROOT / "arc3-qwen36-runtime-wheels",
                            INPUT_ROOT / "datasets" / "jcole75" / "arc3-qwen36-runtime-wheels")
           if p.is_dir()]
print("=== cu13 wheelhouse ===", _wheels)
_envdirs = [str(p) for p in (
    Path("/kaggle/input/competitions/arc-prize-2026-arc-agi-3/environment_files"),
    Path("/kaggle/input/arc-prize-2026-arc-agi-3/environment_files")) if p.is_dir()]
print("=== competition environment_files ===", _envdirs)
print("=== input models dir (should NOT exist) ===", (INPUT_ROOT / "models").exists())
_smi = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
print((_smi.stdout or "").strip()[:900], flush=True)


In [ ]:
# ================= Phase 1 — ASSEMBLE: their runtime + symlink-union model view =================
# Pins below are copied verbatim from sonpham's kaggle_flashnext_setup.py
# (part-A source-bundle). Their preconverted wrapper verifies a PRIVATE Kaggle
# MODEL mount, so we assemble the identical flat view from the three public
# serving-part datasets and reuse everything else of their serve chain.
EXPECTED_RUNTIME_SHA256 = "c06a78d59a74ac278dc2278d26dde6c70c48a4e28bb91fd4fbbefff4484e10f3"
EXPECTED_ZSTD_SHA256 = "7c5468b370f7c47eda07281e3437fafc568f95d10420051e3aa522709f9342c5"
EXPECTED_VERSIONS_LINE = "0.1.dev20073+g8e685d198 2.13.0+cu130 5.15.1 13.0"
RUNTIME_ARCHIVE = "flashnext-gcp-container-site-packages.tar.zst"
QWEN_SERVED_MODEL_NAME = "RadixArk/Qwen3.8-Flash-Next-NVFP4"
MODEL_REVISION = "7b719225242aacd3dbd3f9407468c2ee9a9d2594"
INPUT_ROOT = Path("/kaggle/input")
CU13_HOME = globals().get("CU13_HOME") or {"value": None}

# scratch root: the extracted runtime is ~15 GB — keep it OFF /kaggle/working
# (its ~20 GB doubles as the preserved-output volume).
def _pick_scratch():
    best, best_free = Path("/tmp"), 0
    for cand in (Path("/kaggle/tmp"), Path("/kaggle/temp"), Path("/tmp")):
        try:
            cand.mkdir(parents=True, exist_ok=True)
            free = shutil.disk_usage(str(cand)).free
        except Exception:
            continue
        if free > best_free:
            best, best_free = cand, free
    return best, best_free


SCRATCH_ROOT, _scratch_free = _pick_scratch()
RUNTIME_ROOT = SCRATCH_ROOT / "flashnext-gcp-runtime"
SITE_PACKAGES = RUNTIME_ROOT / "dist-packages"
MODEL_DIR = SCRATCH_ROOT / "flashnext-model"
QWEN_MODEL_PATH = MODEL_DIR
print(f"flashnext-smoke: scratch root {SCRATCH_ROOT} (free {_scratch_free / 1e9:.1f} GB)", flush=True)
RESULTS["meta"]["scratch_root"] = str(SCRATCH_ROOT)


def sha256_file(path):
    digest = __import__("hashlib").sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(16 * 1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def serving_env():
    # VERBATIM from their kaggle_flashnext_setup.py serving_env() — including
    # VLLM_PLE_CPU_OFFLOAD=1 (the single-GPU mechanism: ~104 GB of PLE n-gram
    # tables live in host RAM) and TORCH_CUDA_ARCH_LIST=12.0f (Blackwell).
    env = os.environ.copy()
    current_pythonpath = env.get("PYTHONPATH", "")
    env["PYTHONPATH"] = (
        str(SITE_PACKAGES)
        if not current_pythonpath
        else f"{SITE_PACKAGES}{os.pathsep}{current_pythonpath}"
    )
    packaged_library_dirs = sorted(
        str(path) for path in (SITE_PACKAGES / "nvidia").glob("*/lib") if path.is_dir()
    )
    system_library_dirs = [
        "/usr/local/nvidia/lib64",
        "/usr/local/cuda/lib64",
        "/usr/local/nvidia/lib",
    ]
    current_ld = [entry for entry in env.get("LD_LIBRARY_PATH", "").split(os.pathsep) if entry]
    env["LD_LIBRARY_PATH"] = os.pathsep.join(
        dict.fromkeys(packaged_library_dirs + system_library_dirs + current_ld)
    )
    current_path = [entry for entry in env.get("PATH", "").split(os.pathsep) if entry]
    env["PATH"] = os.pathsep.join(
        dict.fromkeys(["/usr/local/nvidia/bin", "/usr/local/cuda/bin"] + current_path)
    )
    env.update(
        {
            "USE_TF": "0",
            "TRANSFORMERS_NO_TF": "1",
            "TRANSFORMERS_NO_TORCHVISION": "1",
            "VLLM_NO_USAGE_STATS": "1",
            "VLLM_ENABLE_CUDA_COMPATIBILITY": "0",
            "VLLM_PLE_CPU_OFFLOAD": "1",
            "VLLM_PLE_OFFLOAD_READY_TIMEOUT": "1800",
            "PYTORCH_ALLOC_CONF": "expandable_segments:False",  # v2: True needs pidfd_getfd CUDA-IPC, blocked by Kaggle seccomp (v1 boot death in the PLE offload worker)
            "HF_HUB_OFFLINE": "1",
        }
    )
    # v3: "12.0f" made the tarball's flashinfer filter out every major-12 arch
    # ("No supported CUDA architectures found for major versions [12]" in the
    # sm120 fused-MoE JIT). Ladder now controls the arch env: None = unset
    # (flashinfer derives SM 12.0 from the device), or an explicit string.
    env.pop("TORCH_CUDA_ARCH_LIST", None)
    arch = ARCH_OVERRIDE.get("value")
    if arch:
        env["TORCH_CUDA_ARCH_LIST"] = arch
    # v4: flashinfer's sm120 fused-MoE JIT dropped every major-12 arch because
    # the nvcc it found (the image's old /usr/local/cuda) predates SM 12.0.
    # Point the whole toolchain at the CUDA-13.3 pip wheels installed at
    # assemble time (the jcole75 wheelhouse recipe: nvidia/cu13 layout).
    cu13 = CU13_HOME.get("value")
    if cu13:
        env["CUDA_HOME"] = cu13
        env["CUDA_PATH"] = cu13
        env["FLASHINFER_NVCC"] = str(Path(cu13) / "bin" / "nvcc")
        env["FLASHINFER_EXTRA_LDFLAGS"] = f"-L{Path(cu13) / 'lib'} -L/usr/local/nvidia/lib64"
        env["PATH"] = f"{Path(cu13) / 'bin'}{os.pathsep}" + env.get("PATH", "")
        env["LD_LIBRARY_PATH"] = f"{Path(cu13) / 'lib'}{os.pathsep}/usr/local/nvidia/lib64{os.pathsep}" + env.get("LD_LIBRARY_PATH", "")
        env["LIBRARY_PATH"] = f"{Path(cu13) / 'lib'}{os.pathsep}/usr/local/nvidia/lib64{os.pathsep}" + env.get("LIBRARY_PATH", "")
        env["CPATH"] = f"{Path(cu13) / 'include'}{os.pathsep}" + env.get("CPATH", "")
    return env


CU13_HOME = globals().get("CU13_HOME") or {"value": None}


ARCH_OVERRIDE = {"value": None}


RUNTIME_OK = False
ASSEMBLE = {}
try:
    _t0 = time.time()
    # ---- locate mounts (any nesting: /kaggle/input/<slug> or /kaggle/input/datasets/<user>/<slug>)
    _shards = {}
    for _name in ("serving-part-000", "serving-part-001", "serving-part-002"):
        _hits = sorted({p.resolve() for p in INPUT_ROOT.rglob(_name) if p.is_dir()})
        if len(_hits) != 1:
            raise FileNotFoundError(f"expected exactly one mounted {_name}, got {_hits}")
        _shards[_name] = _hits[0]
    _bundles = sorted({p.parent.resolve() for p in INPUT_ROOT.rglob("source-bundle/zstd")})
    if len(_bundles) != 1:
        raise FileNotFoundError(f"expected exactly one source-bundle, got {_bundles}")
    BUNDLE_DIR = _bundles[0]
    _archives = sorted({p.resolve() for p in INPUT_ROOT.rglob(RUNTIME_ARCHIVE)})
    if not _archives:
        raise FileNotFoundError(f"runtime archive {RUNTIME_ARCHIVE} not mounted")
    _preferred = [p for p in _archives if "runtime-exact" in str(p)]
    RUNTIME_TARBALL = (_preferred or _archives)[0]
    ASSEMBLE["shard_dirs"] = {k: str(v) for k, v in _shards.items()}
    ASSEMBLE["bundle_dir"] = str(BUNDLE_DIR)
    ASSEMBLE["runtime_tarball"] = str(RUNTIME_TARBALL)
    print("flashnext-smoke: shards", ASSEMBLE["shard_dirs"], flush=True)

    # ---- v4: CUDA-13.3 compiler toolchain for the flashinfer sm120 JIT ----
    # The tarball may or may not carry an nvcc; the image's is too old for
    # SM 12.0. Prefer a cu13 nvcc found inside the extracted runtime, else
    # install the pinned CUDA-13.3 wheels from the jcole75 wheelhouse.
    def _wire_cu13():
        hits = sorted(RUNTIME_ROOT.rglob("nvidia/cu13/bin/nvcc")) if RUNTIME_ROOT.exists() else []
        if hits:
            CU13_HOME["value"] = str(hits[0].parent.parent)
            print("flashnext-smoke: cu13 nvcc found in runtime:", hits[0], flush=True)
            return
        wh = None
        for cand in (INPUT_ROOT / "arc3-qwen36-runtime-wheels",
                     INPUT_ROOT / "datasets" / "jcole75" / "arc3-qwen36-runtime-wheels"):
            if (cand / "wheels").is_dir():
                wh = cand / "wheels"
                break
        if wh is None:
            print("flashnext-smoke: WARNING no cu13 nvcc and no wheelhouse mount — JIT will fail", flush=True)
            return
        target = SCRATCH_ROOT / "cu13-site"
        target.mkdir(parents=True, exist_ok=True)
        cmd = [sys.executable, "-m", "pip", "install", "--no-index", "--find-links", str(wh),
               "--target", str(target), "--no-deps", "--disable-pip-version-check", "--no-warn-conflicts",
               "nvidia-cuda-nvcc==13.3.73", "nvidia-cuda-crt==13.3.73", "nvidia-cuda-runtime==13.3.29",
               "nvidia-cuda-cccl==13.3.3.4.1", "nvidia-cuda-nvrtc==13.3.33", "nvidia-nvvm==13.3.73",
               "nvidia-curand==10.4.3.29", "nvidia-cublas==13.3.0.5"]
        print("flashnext-smoke: installing cu13 toolchain:", " ".join(cmd[-8:]), flush=True)
        subprocess.run(cmd, check=True)
        nvcc = target / "nvidia" / "cu13" / "bin" / "nvcc"
        if nvcc.is_file():
            CU13_HOME["value"] = str(nvcc.parent.parent)
            out = subprocess.run([str(nvcc), "--version"], capture_output=True, text=True)
            print("flashnext-smoke: cu13 nvcc:", (out.stdout or out.stderr).strip().splitlines()[-1], flush=True)
        else:
            print("flashnext-smoke: WARNING cu13 wheels installed but no bin/nvcc at", nvcc, flush=True)

    _wire_cu13()
    ASSEMBLE["cu13_home"] = CU13_HOME["value"]
    print("flashnext-smoke: bundle", BUNDLE_DIR, "| tarball", RUNTIME_TARBALL, flush=True)

    # ---- their runtime install, verbatim semantics (sha-pinned tarball + zstd) ----
    _sha = sha256_file(RUNTIME_TARBALL)
    if _sha != EXPECTED_RUNTIME_SHA256:
        raise RuntimeError(f"runtime archive drift: {_sha} != {EXPECTED_RUNTIME_SHA256}")
    _zstd_src = BUNDLE_DIR / "zstd"
    _zsha = sha256_file(_zstd_src)
    if _zsha != EXPECTED_ZSTD_SHA256:
        raise RuntimeError(f"bundled zstd drift: {_zsha} != {EXPECTED_ZSTD_SHA256}")
    _tools = SCRATCH_ROOT / "packaging-tools"
    _tools.mkdir(parents=True, exist_ok=True)
    ZSTD_TOOL = _tools / "zstd"
    shutil.copy2(_zstd_src, ZSTD_TOOL)
    ZSTD_TOOL.chmod(0o755)
    shutil.rmtree(RUNTIME_ROOT, ignore_errors=True)
    RUNTIME_ROOT.mkdir(parents=True)
    _tx = time.time()
    subprocess.run(["tar", f"--use-compress-program={ZSTD_TOOL}", "-xf",
                    str(RUNTIME_TARBALL), "-C", str(RUNTIME_ROOT)], check=True)
    ASSEMBLE["extract_s"] = round(time.time() - _tx, 1)
    if not (SITE_PACKAGES / "vllm").is_dir():
        raise RuntimeError(f"extracted runtime incomplete: {SITE_PACKAGES}")
    _probe = subprocess.run(
        [sys.executable, "-c",
         ("import platform,torch,transformers,vllm; "
          "print(platform.python_version(),vllm.__version__,torch.__version__,"
          "transformers.__version__,torch.version.cuda)")],
        env=serving_env(), capture_output=True, text=True)
    ASSEMBLE["runtime_import"] = (_probe.stdout or "").strip()[:200]
    print("flashnext-smoke: exact runtime import:", ASSEMBLE["runtime_import"], flush=True)
    if _probe.returncode:
        raise RuntimeError(f"exact runtime import failed:\n{(_probe.stderr or '')[-3000:]}")
    if EXPECTED_VERSIONS_LINE not in (_probe.stdout or ""):
        raise RuntimeError(f"exact serving versions drifted: {ASSEMBLE['runtime_import']}")

    # ---- symlink-union model view (their zero-copy layout, dataset-mount flavour) ----
    shutil.rmtree(MODEL_DIR, ignore_errors=True)
    MODEL_DIR.mkdir(parents=True)
    _seen = {}
    for _name in sorted(_shards):
        for _member in sorted(_shards[_name].iterdir()):
            if not _member.is_file():
                raise RuntimeError(f"unexpected nested entry in {_name}: {_member}")
            if _member.name in _seen:
                raise RuntimeError(f"duplicate file across shards: {_member.name} "
                                   f"({_seen[_member.name]} vs {_name})")
            _seen[_member.name] = _name
            if _member.name == "model.safetensors.index.json":
                shutil.copy2(_member, MODEL_DIR / _member.name)
            else:
                (MODEL_DIR / _member.name).symlink_to(_member)
    _st_files = sorted(MODEL_DIR.glob("*.safetensors"))
    _st_bytes = sum(p.stat().st_size for p in _st_files)
    ASSEMBLE["model_files"] = len(_seen)
    ASSEMBLE["safetensor_files"] = len(_st_files)
    ASSEMBLE["safetensor_bytes"] = _st_bytes
    if len(_st_files) != 206:
        raise RuntimeError(f"expected 206 safetensors, got {len(_st_files)}")
    if _st_bytes < 186000000000:
        raise RuntimeError(f"payload too small: {_st_bytes}")
    _index = json.loads((MODEL_DIR / "model.safetensors.index.json").read_text())
    _needed = sorted(set((_index.get("weight_map") or {}).values()))
    _missing = [n for n in _needed if not (MODEL_DIR / n).is_file()]
    ASSEMBLE["index_files_referenced"] = len(_needed)
    if _missing:
        raise RuntimeError(f"index references missing files: {_missing[:8]} "
                           f"(+{max(0, len(_missing) - 8)} more)")
    if list(MODEL_DIR.glob("model-plefp8-*.safetensors")):
        raise RuntimeError("serving view still contains FP8 PLE source files")

    # ---- PLE-conversion provenance vs the GCP winner (their check, hashes-by-manifest) ----
    _expected = json.loads((BUNDLE_DIR / "FLASHNEXT_GCP_MODEL_INFO.json").read_text())["ple_conversion"]
    _observed = json.loads((MODEL_DIR / "ple-bf16-conversion.json").read_text())
    _exp_files = {i["target"]: (int(i["target_bytes"]), i["target_sha256"]) for i in _expected["files"]}
    _obs_files = {i["target"]: (int(i["target_bytes"]), i["target_sha256"]) for i in _observed["files"]}
    if _obs_files != _exp_files or _observed.get("index_sha256") != _expected.get("index_sha256"):
        raise RuntimeError("PLE conversion provenance differs from the GCP winner")
    for _tname, (_tbytes, _sha_unused) in _exp_files.items():
        if (MODEL_DIR / _tname).stat().st_size != _tbytes:
            raise RuntimeError(f"PLE file size drift: {_tname}")
    ASSEMBLE["ple_files_verified"] = len(_exp_files)
    _cfg = json.loads((MODEL_DIR / "config.json").read_text())
    ASSEMBLE["architectures"] = _cfg.get("architectures")
    ASSEMBLE["model_type"] = _cfg.get("model_type")
    ASSEMBLE["assemble_s"] = round(time.time() - _t0, 1)
    RUNTIME_OK = True
    print(f"flashnext-smoke: ASSEMBLE OK in {ASSEMBLE['assemble_s']} s — "
          f"{len(_st_files)} safetensors / {_st_bytes / 1e9:.1f} GB, "
          f"arch {ASSEMBLE['architectures']}, elapsed {elapsed_min():.1f} min", flush=True)
except Exception as _exc:
    traceback.print_exc()
    ASSEMBLE["error"] = repr(_exc)[:800]
    RESULTS["verdicts"]["boot"] = "ASSEMBLE-FAILED: " + repr(_exc)[:300]
    print("flashnext-smoke: ASSEMBLE FAILED — every later phase will be skipped", flush=True)
RESULTS["meta"]["assemble"] = ASSEMBLE
RESULTS["meta"]["model_path"] = str(QWEN_MODEL_PATH)
RESULTS["meta"]["served_model_name"] = QWEN_SERVED_MODEL_NAME
host_snapshot("after_assemble")
save_results()


In [ ]:
# ================= serve library — helpers + their launch argv (gate-proven) =================
# Minimal slice of the arc3-flashnext-gate serve chain: constants, HTTP/log/GPU
# helpers, and the server lifecycle (their serving_env + argv verbatim). The
# gate's load-generator/battery phases are NOT included — the load here is the
# real duck harness.
VLLM_HOST = "127.0.0.1"
VLLM_PORT = 1234
VLLM_ROOT = f"http://{VLLM_HOST}:{VLLM_PORT}"
VLLM_API = VLLM_ROOT + "/v1"
VLLM_MAX_MODEL_LEN = 32768   # their launch_server value

CURRENT_SERVER = {"proc": None, "log": str(WORKING_DIR / "vllm-gcp_exact.log"),
                  "tag": "none", "flags": []}


def http_json(url, payload=None, timeout=120):
    data = None if payload is None else json.dumps(payload).encode("utf-8")
    req = urllib.request.Request(url, data=data, headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(req, timeout=timeout) as resp:
        return json.loads(resp.read().decode("utf-8"))


def server_alive(timeout=5):
    try:
        http_json(VLLM_API + "/models", timeout=timeout)
        return True
    except Exception:
        return False


def vllm_procs():
    out = subprocess.run(["pgrep", "-f", "vllm.entrypoints"], capture_output=True, text=True)
    return [int(x) for x in out.stdout.split() if x.strip().isdigit()]


def gpu_sample():
    try:
        out = subprocess.run(
            ["nvidia-smi", "--query-gpu=utilization.gpu,memory.used",
             "--format=csv,noheader,nounits"],
            capture_output=True, text=True, timeout=20)
        util, mem = out.stdout.strip().splitlines()[0].split(",")
        return {"util_pct": int(util.strip()), "mem_mib": int(mem.strip())}
    except Exception:
        return None


def tail_log_lines(path, max_bytes=524288):
    p = Path(path)
    if not p.exists():
        return []
    with p.open("rb") as handle:
        handle.seek(0, 2)
        size = handle.tell()
        handle.seek(max(0, size - max_bytes))
        return handle.read().decode("utf-8", errors="replace").splitlines()


# ---- their launch_server argv VERBATIM (kaggle_flashnext_setup.py, kv "auto") ----
FLASH_SERVE_FLAGS = [
    "--model", str(QWEN_MODEL_PATH),
    "--served-model-name", QWEN_SERVED_MODEL_NAME,
    "--host", VLLM_HOST,
    "--port", str(VLLM_PORT),
    "--tensor-parallel-size", "1",
    "--distributed-executor-backend", "mp",
    "--gpu-memory-utilization", "0.96",
    "--max-model-len", "32768",
    "--max-num-seqs", "22",
    "--max-num-batched-tokens", "6144",
    "--kv-cache-dtype", "auto",
    "--enable-prefix-caching",
    "--no-enable-flashinfer-autotune",
    "--enable-auto-tool-choice",
    "--tool-call-parser", "qwen3_xml",
    "--generation-config", "vllm",
    "--default-chat-template-kwargs", '{"preserve_thinking": true}',
    "--reasoning-parser", "qwen3",
]
# pre-registered retry rung if their exact config fails (OOM or otherwise):
REDUCED_SERVE_FLAGS = [
    "--model", str(QWEN_MODEL_PATH),
    "--served-model-name", QWEN_SERVED_MODEL_NAME,
    "--host", VLLM_HOST,
    "--port", str(VLLM_PORT),
    "--tensor-parallel-size", "1",
    "--distributed-executor-backend", "mp",
    "--gpu-memory-utilization", "0.92",
    "--max-model-len", "16384",
    "--max-num-seqs", "12",
    "--max-num-batched-tokens", "6144",
    "--kv-cache-dtype", "auto",
    "--enable-prefix-caching",
    "--no-enable-flashinfer-autotune",
    "--enable-auto-tool-choice",
    "--tool-call-parser", "qwen3_xml",
    "--generation-config", "vllm",
    "--default-chat-template-kwargs", '{"preserve_thinking": true}',
    "--reasoning-parser", "qwen3",
]
# (tag, flags, TORCH_CUDA_ARCH_LIST override): None = unset -> flashinfer
# derives SM 12.0 from the device; "12.0a" = the CUTLASS sm120a spelling.
BOOT_LADDER = [("gcp_exact", FLASH_SERVE_FLAGS, None),
               ("gcp_exact_arch120a", FLASH_SERVE_FLAGS, "12.0a"),
               ("reduced", REDUCED_SERVE_FLAGS, None)]


# ---- server lifecycle (their serving_env, own process group) --------------------
ENGINE_LINE_PATTERNS = {
    "engine_config": "Initializing a V1 LLM engine",
    "non_default_args": "non-default args",
    "attention_backend": "attention backend",
    "kv_cache_size": "GPU KV cache size",
    "max_concurrency": "Maximum concurrency for",
    "kv_cache_memory": "Available KV cache memory",
    "model_load": "Model loading took",
    "ple_lower": "ple",
    "offload": "offload",
    "cuda_oom": "CUDA out of memory",
    "torch_oom": "OutOfMemoryError",
    "cuda_error": "CUDA error",
    "illegal_memory": "illegal memory access",
    "engine_dead": "EngineDeadError",
    "traceback": "Traceback (most recent call last)",
}


def capture_engine_lines(log_path, max_hits=4):
    lines = tail_log_lines(log_path, max_bytes=4_000_000)
    found = {}
    for key, needle in ENGINE_LINE_PATTERNS.items():
        if key == "ple_lower":
            hits = [ln.strip()[:1200] for ln in lines
                    if ("ple" in ln.lower() and ("offload" in ln.lower() or "PLE" in ln))]
        else:
            hits = [ln.strip()[:1200] for ln in lines if needle in ln]
        if hits:
            found[key] = {"count": len(hits), "lines": hits[:max_hits]}
    backend = None
    for ln in lines:
        m = re.search(r"Using (\S+) attention backend", ln)
        if m and backend is None:
            backend = m.group(1)
    found["attention_backend_name"] = backend
    found["oomish"] = bool(found.get("cuda_oom") or found.get("torch_oom"))
    return found


def print_engine_lines(tag, found):
    print(f"flashnext-smoke: [{tag}] engine lines:", flush=True)
    for key in ("engine_config", "attention_backend", "kv_cache_size", "max_concurrency",
                "kv_cache_memory", "model_load", "ple_lower", "offload"):
        for ln in (found.get(key) or {}).get("lines", [])[:2]:
            print(f"  {key}: {ln[:1100]}", flush=True)
    print(f"  attention backend : {found.get('attention_backend_name')}", flush=True)
    for key in ("cuda_oom", "torch_oom", "cuda_error", "illegal_memory", "engine_dead"):
        for ln in (found.get(key) or {}).get("lines", [])[:2]:
            print(f"  !! {key}: {ln[:600]}", flush=True)


def stop_server(reason):
    print(f"flashnext-smoke: stopping vLLM ({reason})", flush=True)
    proc = CURRENT_SERVER.get("proc")
    if proc is not None and proc.poll() is None:
        try:
            os.killpg(os.getpgid(proc.pid), signal.SIGTERM)
        except Exception:
            pass
    subprocess.run(["pkill", "-TERM", "-f", "vllm.entrypoints"], check=False)
    deadline = time.time() + 90
    while time.time() < deadline and vllm_procs():
        time.sleep(3)
    if vllm_procs():
        if proc is not None and proc.poll() is None:
            try:
                os.killpg(os.getpgid(proc.pid), signal.SIGKILL)
            except Exception:
                pass
        subprocess.run(["pkill", "-9", "-f", "vllm.entrypoints"], check=False)
        time.sleep(10)
    deadline = time.time() + 240
    while time.time() < deadline:
        sample = gpu_sample()
        if sample is not None and sample["mem_mib"] < 8000:
            break
        time.sleep(5)
    CURRENT_SERVER["proc"] = None
    print(f"flashnext-smoke: server stopped, gpu={gpu_sample()}", flush=True)


def served_ids():
    try:
        return [m.get("id") for m in http_json(VLLM_API + "/models", timeout=5).get("data", [])]
    except Exception:
        return None


def start_server(flags, tag, timeout_s=1800):
    """Boot the pinned vLLM with the given full argv in THEIR serving_env().
    1800 s readiness = their wait_server + VLLM_PLE_OFFLOAD_READY_TIMEOUT
    number. Raises on death/timeout with the log tail printed; the boot is
    recorded under RESULTS['boots'][tag] either way."""
    log_path = WORKING_DIR / f"vllm-{tag}.log"
    cmd = [sys.executable, "-m", "vllm.entrypoints.openai.api_server", *flags]
    print(f"flashnext-smoke: starting vLLM [{tag}] (elapsed {elapsed_min():.1f} min):",
          " ".join(cmd), flush=True)
    handle = log_path.open("w", encoding="utf-8")
    t0 = time.time()
    proc = subprocess.Popen(cmd, env=serving_env(), stdout=handle, stderr=subprocess.STDOUT,
                            text=True, start_new_session=True)
    CURRENT_SERVER.update({"proc": proc, "log": str(log_path), "tag": tag, "flags": flags})
    boot = {"tag": tag, "flags": flags, "started_utc": datetime.utcnow().isoformat() + "Z"}
    RESULTS["boots"][tag] = boot
    deadline = time.time() + timeout_s
    while time.time() < deadline:
        if proc.poll() is not None:
            lines = tail_log_lines(log_path)
            print(f"flashnext-smoke: [{tag}] BOOT FAILURE rc={proc.returncode} — log tail:", flush=True)
            print("\n".join(lines[-150:]), flush=True)
            boot.update({"ok": False, "rc": proc.returncode, "boot_s": round(time.time() - t0, 1),
                         "log_tail": lines[-60:], "engine_lines": capture_engine_lines(log_path)})
            print_engine_lines(tag, boot["engine_lines"])
            save_results()
            raise RuntimeError(f"vLLM [{tag}] died during startup rc={proc.returncode}")
        if server_alive():
            ids = served_ids()
            found = capture_engine_lines(log_path)
            boot.update({"ok": True, "boot_s": round(time.time() - t0, 1), "served_ids": ids,
                         "engine_lines": found, "gpu_after": gpu_sample(),
                         "host_after": host_snapshot(f"after_boot_{tag}")})
            print(f"flashnext-smoke: vLLM ready [{tag}] in {boot['boot_s']} s, served ids {ids}",
                  flush=True)
            print_engine_lines(tag, found)
            print(f"flashnext-smoke: [{tag}] gpu after boot {boot['gpu_after']} | "
                  f"host {boot['host_after']}", flush=True)
            if ids != [QWEN_SERVED_MODEL_NAME]:
                print(f"flashnext-smoke: WARNING served ids {ids} != [{QWEN_SERVED_MODEL_NAME}]",
                      flush=True)
            save_results()
            return flags
        time.sleep(10)
    lines = tail_log_lines(log_path)
    print(f"flashnext-smoke: [{tag}] BOOT TIMEOUT — log tail:", flush=True)
    print("\n".join(lines[-150:]), flush=True)
    boot.update({"ok": False, "rc": None, "timeout": True, "boot_s": round(time.time() - t0, 1),
                 "log_tail": lines[-60:], "engine_lines": capture_engine_lines(log_path)})
    save_results()
    raise TimeoutError(f"vLLM [{tag}] not ready in {timeout_s}s")


def reset_prefix_cache():
    # vLLM API server: POST /reset_prefix_cache. Best effort — recorded, never fatal.
    try:
        req = urllib.request.Request(VLLM_ROOT + "/reset_prefix_cache", data=b"", method="POST")
        with urllib.request.urlopen(req, timeout=30) as resp:
            return resp.status
    except Exception as exc:
        return ("failed: " + repr(exc))[:120]




In [ ]:
# ===================== Phase 2 — BOOT (their argv verbatim; pre-registered retry rung) =====================
BOOT_TAG = None
try:
    if not RUNTIME_OK:
        raise RuntimeError("assemble failed — boot skipped")
    for _tag, _flags, _arch in BOOT_LADDER:
        try:
            ARCH_OVERRIDE["value"] = _arch
            print(f"flashnext-smoke: rung [{_tag}] TORCH_CUDA_ARCH_LIST={_arch!r}", flush=True)
            start_server(_flags, tag=_tag)
            BOOT_TAG = _tag
            break
        except Exception as _exc:
            traceback.print_exc()
            RESULTS["boots"].setdefault(_tag, {})["error"] = repr(_exc)[:600]
            print(f"flashnext-smoke: boot [{_tag}] FAILED — next rung of the ladder", flush=True)
            save_results()
            stop_server(f"cleanup after failed {_tag}")
    if BOOT_TAG is None:
        RESULTS["verdicts"]["boot"] = "BOOT FAILED on all rungs (device-detect arch, 12.0a, reduced)"
    else:
        _b = RESULTS["boots"][BOOT_TAG]
        _el = _b.get("engine_lines") or {}
        RESULTS["verdicts"]["boot"] = (
            f"BOOT OK on rung '{BOOT_TAG}'"
            + (" (DEGRADED-CONFIG: their exact config did not serve)" if BOOT_TAG == "reduced" else "")
            + f" in {_b.get('boot_s')} s — attention backend {_el.get('attention_backend_name')}, "
            + f"gpu after boot {_b.get('gpu_after')}")
except Exception as _exc:
    traceback.print_exc()
    RESULTS["verdicts"].setdefault("boot", "BOOT PHASE ERROR: " + repr(_exc)[:300])
print("flashnext-smoke: BOOT VERDICT:", RESULTS["verdicts"].get("boot"), flush=True)
save_results()


# The smoke is unreadable without a live server — die loudly here rather than
# spend three hours playing games against a dead endpoint.
if BOOT_TAG is None:
    raise RuntimeError("Flash-Next server did not boot on any rung — smoke aborted")


In [ ]:
# ============ duck analyzer env — the ONLY wiring change vs the 27B kernel ============
# v3: the gate ASSEMBLE cell rebinds BUNDLE_DIR to sonpham's source-bundle
# (their June-fork tree, which carries its own solver pkls WITHOUT
# the anim fields — v2 loaded it and died on hard_noop_guard). Re-pin the
# TAAF bundle before the deploy cell reads any pkl, and drop their tree from
# sys.path + sys.modules so the anim classes win.
BUNDLE_DIR = TAAF_BUNDLE_DIR
_sonpham_paths = [p for p in sys.path if "source-bundle" in p or "sonphamorg" in p]
for _p in _sonpham_paths:
    sys.path.remove(_p)
for _m, _mod in list(sys.modules.items()):
    if _m.split(".")[0] in ("inference", "taaf"):
        _f = getattr(_mod, "__file__", "") or ""
        if "source-bundle" in _f or "sonphamorg" in _f:
            del sys.modules[_m]
print("duck-env: BUNDLE_DIR re-pinned to", BUNDLE_DIR, "| pruned", len(_sonpham_paths), "paths")
assert (BUNDLE_DIR / "taaf-kaggle-bundle.json").is_file()
assert "anim-20260807" in str(BUNDLE_DIR), f"re-pin failed: {BUNDLE_DIR}"
# Reproduces the 27B bundle's exported setup_env EXACTLY (same keys,
# same stock sampling: temp 0.6 / top-p 0.95 / top-k 20 / thinking on) except:
#   * base URL + model id -> the Flash-Next server booted above (port 1234,
#     served name from their FLASH_SERVE_FLAGS);
#   * LOCAL_ANALYZER_CONTEXT_WINDOW 24576 + LOCAL_ANALYZER_MAX_OUTPUT 4096:
#     their server is --max-model-len 32768; the duck's default 32768 window
#     with no max_tokens could push prompt+completion past the server cap.
#     (The 27B baseline ran a 32768 window on a 65536-ctx server. Noted as a
#     deviation in the README; the stock 27B effective history is 4-9 turns.)
#   * PYTHONPATH NOT exported: the cu130 serving runtime must not shadow the
#     notebook's harness deps. The harness talks plain HTTP via `requests`;
#     only the vLLM server subprocess sees the extracted site-packages.
# These exports MUST land before the deploy pkls are loaded two cells down:
# tool_agent reads CONTEXT_WINDOW/MAX_OUTPUT at import time.
if BOOT_TAG is None:
    raise RuntimeError("no Flash-Next server — cannot wire the analyzer")
_booted_flags = CURRENT_SERVER["flags"]
_booted_ctx = int(_booted_flags[_booted_flags.index("--max-model-len") + 1])
if _booted_ctx >= 32768:
    _ctx_window, _max_out = "24576", "4096"
else:
    # reduced rung (16k server): shrink the window proportionally
    _ctx_window, _max_out = "12288", "2048"

duck_env = {
    "USE_TF": "0",
    "TRANSFORMERS_NO_TF": "1",
    "TRANSFORMERS_NO_TORCHVISION": "1",
    "VLLM_NO_USAGE_STATS": "1",
    "LOCAL_ANALYZER_BASE_URL": VLLM_API,
    "OPENAI_BASE_URL": VLLM_API,
    "LOCAL_ANALYZER_PROVIDER": "vllm",
    "OPENAI_PROVIDER": "vllm",
    "LOCAL_ANALYZER_MODEL_ID": QWEN_SERVED_MODEL_NAME,
    "INFERENCE_ANALYZER_MODEL": QWEN_SERVED_MODEL_NAME,
    "LOCAL_ANALYZER_APP_NAME": "ARC3 Agent Harness",
    "LOCAL_ANALYZER_CONTEXT_WINDOW": _ctx_window,
    "LOCAL_ANALYZER_MAX_OUTPUT": _max_out,
    "LOCAL_ANALYZER_TOOL_STEPS": "0",
    "LOCAL_ANALYZER_TOOL_TIMEOUT": "30",
    "LOCAL_ANALYZER_TOOL_OUTPUT_TOKENS": "1024",
    "LOCAL_ANALYZER_YIELD_SECONDS": "60",
    "LOCAL_ANALYZER_TEMPERATURE": "0.6",
    "LOCAL_ANALYZER_TOP_P": "0.95",
    "LOCAL_ANALYZER_TOP_K": "20",
    "LOCAL_ANALYZER_ENABLE_THINKING": "true",
    "MULTIMODAL_CONTEXT": "current_grid",
    "MULTIMODAL_UPSCALE": "4",
}
os.environ.update(duck_env)
SETUP_ENV_PATH.write_text(json.dumps(duck_env, indent=2, sort_keys=True) + "\n", encoding="utf-8")


def _run_shell_commands(filename, *, label, check):
    # Teardown shim: the v12 run cell calls this with teardown_commands.json,
    # which targets the 27B bundle's own vLLM pid file. This kernel's server
    # is ours — stop it directly and flush the results sink.
    print(f"taaf.kaggle: {label} ({filename}) -> flashnext stop_server", flush=True)
    try:
        stop_server(label)
    except Exception as exc:  # noqa: BLE001
        print("flashnext-smoke: teardown stop failed:", repr(exc)[:300], flush=True)
    save_results()


assert os.environ.get("INFERENCE_ANALYZER_MODEL") == QWEN_SERVED_MODEL_NAME
RESULTS["meta"]["duck_env"] = {k: v for k, v in duck_env.items()
                               if k.startswith(("LOCAL_", "INFERENCE_", "OPENAI_", "MULTIMODAL_"))}
save_results()
print("\n✅ duck analyzer wired to Flash-Next")
print("Analyzer endpoint:", os.environ["LOCAL_ANALYZER_BASE_URL"])
print("Analyzer model:   ", os.environ["INFERENCE_ANALYZER_MODEL"])
print("Context window:   ", _ctx_window, "| max output:", _max_out,
      "| server max-model-len:", _booted_ctx)


In [ ]:
# Boot attestation (flashnext variant of doctrine v2, 2026-08-17): the
# assembled model view must be the RadixArk Flash-Next NVFP4 package and the
# LIVE server must answer through the duck's analyzer endpoint. Discriminators
# from the model card + the gate v4 Kaggle run: architectures
# Qwen4ExpForConditionalGeneration, model_type qwen4_exp, 206 safetensors
# > 186 GB (NVFP4 experts + BF16-converted PLE tables; the PLE conversion is
# hash-verified against FLASHNEXT_GCP_MODEL_INFO.json in the assemble cell).
# A wrong view must DIE here, before any game action is spent.
import hashlib as _hashlib
import urllib.request as _rq

_cfg_path = QWEN_MODEL_PATH / "config.json"
_cfg_raw = _cfg_path.read_bytes()
_cfg = json.loads(_cfg_raw)
assert _cfg.get("architectures") == ["Qwen4ExpForConditionalGeneration"], (
    f"attest FAIL: architectures {_cfg.get('architectures')}")
assert _cfg.get("model_type") == "qwen4_exp", (
    f"attest FAIL: model_type {_cfg.get('model_type')}")
print("attest: config sha256", _hashlib.sha256(_cfg_raw).hexdigest())

_shards = sorted(QWEN_MODEL_PATH.glob("*.safetensors"))
_total = sum(p.stat().st_size for p in _shards)
print(f"attest: {len(_shards)} shards, {_total} bytes total")
assert len(_shards) == 206, f"attest FAIL: shard count {len(_shards)} != 206"
assert _total > 186_000_000_000, f"attest FAIL: total shard bytes {_total} too small"

# Greedy decode fingerprint through the analyzer endpoint — logged (not
# asserted) for cross-run comparison, and proves the served model answers.
_base = (os.environ.get("LOCAL_ANALYZER_BASE_URL") or "http://127.0.0.1:1234/v1").rstrip("/")
if not _base.endswith("/v1"):
    _base += "/v1"
_body = json.dumps({
    "model": os.environ["INFERENCE_ANALYZER_MODEL"],
    "messages": [{"role": "user", "content": "Reply with exactly the sum of 17 and 25, then the word quack."}],
    "temperature": 0.0,
    "max_tokens": 48,
    "chat_template_kwargs": {"enable_thinking": False, "preserve_thinking": True},
}).encode()
_req = _rq.Request(_base + "/chat/completions", data=_body, headers={
    "Content-Type": "application/json",
    "Authorization": "Bearer " + (os.environ.get("LOCAL_ANALYZER_API_KEY") or "EMPTY"),
})
with _rq.urlopen(_req, timeout=300) as _resp:
    _reply = json.loads(_resp.read())["choices"][0]["message"].get("content") or ""
print("attest: decode fingerprint", repr(_reply)[:160])
print("attest: decode sha256", _hashlib.sha256(_reply.encode()).hexdigest())
print("attest: OK — Flash-Next NVFP4 signature + live analyzer endpoint verified before any game")


In [ ]:
def _soft_end_time(max_runtime_s: float, *, run_as_submission: bool) -> datetime | None:
    if run_as_submission or max_runtime_s <= 0:
        return None
    budget = max(1.0, max_runtime_s)
    buffer = min(SOFT_DEADLINE_BUFFER_S, budget / 2)
    start = datetime.fromtimestamp(NOTEBOOK_START_EPOCH)
    return start + timedelta(seconds=budget - buffer)


def _competition_games():
    import arc_agi

    import taaf.game_api

    spec = taaf.game_api.ArcadeSpec(
        operation_mode=arc_agi.OperationMode.COMPETITION,
        arc_base_url=os.environ.get("ARC_BASE_URL", "http://gateway:8001/"),
        environments_dir="",
    )
    arcade = arc_agi.Arcade(
        operation_mode=arc_agi.OperationMode.COMPETITION,
        arc_base_url=spec.arc_base_url,
        environments_dir="",
    )
    game_ids = [env_info.game_id for env_info in arcade.available_environments]
    if not game_ids:
        raise RuntimeError("Competition Arcade exposed zero environments.")
    return [taaf.game_api.GameAPI(env_name=game_id, arcade_spec=spec) for game_id in game_ids]


@contextlib.contextmanager
def _tee_to_file(log_path: Path):
    log_path.parent.mkdir(parents=True, exist_ok=True)
    log_file = open(log_path, "w", buffering=1)
    original_stdout = sys.stdout
    original_stderr = sys.stderr
    sys.stdout = _Tee(original_stdout, log_file)
    sys.stderr = _Tee(original_stderr, log_file)
    try:
        yield
    finally:
        sys.stdout = original_stdout
        sys.stderr = original_stderr
        log_file.close()


class _Tee:
    def __init__(self, *streams: TextIO) -> None:
        self._streams = streams

    def write(self, data: str) -> int:
        n = 0
        for stream in self._streams:
            n = stream.write(data)
        return n

    def flush(self) -> None:
        for stream in self._streams:
            stream.flush()

    def isatty(self) -> bool:
        return any(getattr(stream, "isatty", lambda: False)() for stream in self._streams)

In [ ]:
true_submission = _env_bool("KAGGLE_IS_COMPETITION_RERUN", False)
run_as_submission = _env_bool("TAAF_RUN_AS_SUBMISSION", False) or true_submission
os.environ["ONLY_RESET_LEVELS"] = "true"
os.environ["TAAF_RUN_AS_SUBMISSION"] = "1" if run_as_submission else "0"
os.environ["TAAF_MINIMAL_DIAGNOSTICS"] = "1" if run_as_submission else "0"

with open(BUNDLE_DIR / "deploy_target.pkl", "rb") as file:
    target = pickle.load(file)
target.actual_run_as_submission = run_as_submission
target.is_competition_rerun = true_submission
soft_end = _soft_end_time(float(getattr(target, "max_runtime_s", 0.0) or 0.0), run_as_submission=run_as_submission)

with open(BUNDLE_DIR / "benchmark_initial.pkl", "rb") as file:
    bm = pickle.load(file)
bm.job_dir = WORKING_DIR

In [ ]:
# Smoke/eval hook: THE MODEL-SWAP READ. A NORMAL COMMIT runs the STOCK duck
# on all 25 public games against the Flash-Next server, eval geometry
# (7920 s per game, concurrency 28 from the serialized solver). One
# phase, no grafts, ONLY_RESET_LEVELS=true, stock sampling. The scored rerun
# path (KAGGLE_IS_COMPETITION_RERUN) never enters this branch.
GAMES_25 = [
    "ar25-0c556536",
    "bp35-0a0ad940",
    "cd82-fb555c5d",
    "cn04-2fe56bfb",
    "dc22-fdcac232",
    "ft09-0d8bbf25",
    "g50t-5849a774",
    "ka59-38d34dbb",
    "lf52-271a04aa",
    "lp85-305b61c3",
    "ls20-9607627b",
    "m0r0-492f87ba",
    "r11l-495a7899",
    "re86-8af5384d",
    "s5i5-18d95033",
    "sb26-7fbdac44",
    "sc25-635fd71a",
    "sk48-d8078629",
    "sp80-589a99af",
    "su15-1944f8ab",
    "tn36-ef4dde99",
    "tr87-cd924810",
    "tu93-0768757b",
    "vc33-5430563c",
    "wa30-ee6fef47"
]
SMOKE_PHASES = [
    ("stock", GAMES_25, 7920, {"TP_ENABLE": "0", "TP2_ENABLE": "0", "TP4_ENABLE": "0",
                                        "TP5_ENABLE": "0", "TP6_ENABLE": "0"}),
    # tuned = neutral Pack-1 base + emission (act-floor, wm-from-reasoning) +
    # explorer fallback + action-economy prompt. Composed arm: the goal is a
    # >=3-LB submission today, attribution later.
    ("tuned", GAMES_25, 7920, {"TP_ENABLE": "1", "TP_TRIM_LOW_WATER": "1.0",
                                        "TP_CONTEXT_WINDOW": "0", "TP_YIELD_SECONDS": "-1",
                                        "TP_TOOL_STEPS": "-1", "TP_BATCH_CAP": "0",
                                        "TP_KEEP_NOTES_ON_GAME_OVER": "1",
                                        "TP2_ENABLE": "0", "TP4_ENABLE": "1",
                                        "TP5_ENABLE": "1", "TP6_ENABLE": "1"}),
]
FN_PHASE_ERRORS = []
FN_ALL_RUNS = []

if not run_as_submission:
    import arc_agi
    from taaf.game_api import ArcadeSpec, GameAPI

    def _resolve_env_dir():
        candidates = [
            Path("/kaggle/input/competitions/arc-prize-2026-arc-agi-3/environment_files"),
            Path("/kaggle/input/arc-prize-2026-arc-agi-3/environment_files"),
        ]
        for cand in candidates:
            if cand.is_dir():
                return str(cand)
        for hit in Path("/kaggle/input").rglob("environment_files"):
            if hit.is_dir():
                return str(hit)
        raise RuntimeError("environment_files dir not found in /kaggle/input")

    _env_dir = _resolve_env_dir()
    _spec = ArcadeSpec(operation_mode=arc_agi.OperationMode.OFFLINE, environments_dir=_env_dir)
    bm.games = [GameAPI(env_name=name, arcade_spec=_spec) for name in SMOKE_PHASES[0][1]]
    bm.n_passes = 1
    bm.game_weights = None
    bm.label = "flashnext-smoke-" + SMOKE_PHASES[0][0]
    bm.solver.max_runtime_s_per_game = float(SMOKE_PHASES[0][2])
    soft_end = datetime.fromtimestamp(NOTEBOOK_START_EPOCH) + timedelta(seconds=18600)
    print(f"smoke hook: phases={[(p[0], len(p[1]), p[2]) for p in SMOKE_PHASES]} "
          f"env_dir={_env_dir} concurrency={bm.solver.concurrency} "
          f"per_game_cap={bm.solver.max_runtime_s_per_game}s soft_end={soft_end}")
else:
    print("scored rerun: smoke hook inert — full competition games")

print("Benchmark analyzer model:", os.environ.get("INFERENCE_ANALYZER_MODEL"))


In [ ]:
# ==== graft install (tuned phase machinery; every flag phase-controlled) ====
# All grafts install ONCE; behaviour is env-gated per phase (TP*_ENABLE).
# Defaults here are ALL OFF — the stock phase runs first.
for _flag in ("TP_ENABLE", "TP2_ENABLE", "TP4_ENABLE", "TP5_ENABLE", "TP6_ENABLE"):
    os.environ[_flag] = "0"
os.environ["TP5_WM_FROM_REASONING"] = "1"
os.environ["TP5_ACT_FLOOR"] = "3"
os.environ["TP4_STALL_T3"] = "30"
os.environ["TP4_BUDGET"] = "800"
os.environ["TP2_STALL_T2"] = "30"
_GRAFT_SOURCES = {
    'graft_throughput.py': '"""Throughput graft (Pack 1, 2026-08-29) — plumbing-only changes that raise\nactions-per-game of the anim-bundle ToolAgent. No prompt text changes.\n\nEvidence: docs/research-2026-08-29/R4-harness-throughput-audit.md — every\nplay dies on the 7,920 s clock at ~88 actions/game; each call re-prefills a\n16-20k-token prompt (prefix-cache hit 0-20%) because the trimmer drops one\nblock per turn; the 60 s yield is shorter than one call (26% of wall in\naction-less slices); carried notes are wiped on every GAME_OVER; blind\n20-140-action batches burn efficiency and trigger GAME_OVERs.\n\nSeams (anim bundle inference/agent/tool_agent.py, the code that plays at eval):\n- _trim_messages_for_context (:2056) — hysteresis: when over budget, cut to\n  TP_TRIM_LOW_WATER x budget in one go so the cached prefix survives turns.\n- __init__ — _context_budget_tokens / _yield_seconds / _tool_steps are set\n  from module constants read at import; override per instance from\n  TP_CONTEXT_WINDOW / TP_YIELD_SECONDS / TP_TOOL_STEPS.\n- _update_summarized_knowledge_from_step_summary (:1343) — wipes the carried\n  notes on game_over; keep them (TP_KEEP_NOTES_ON_GAME_OVER). Level-up and\n  run-complete wipes are untouched.\n- _run_python_tool (:1693) + _normalize_python_actions (:1614) — per tool\n  call action budget (TP_BATCH_CAP): requests beyond the cap are truncated,\n  and a further action() call raises ValueError inside the sandbox so the\n  model sees a readable error instead of a blind 100-action walk.\n\nFlags (read at call time; TP_ENABLE=0 turns every seam into a pass-through):\n  TP_ENABLE=1  TP_TRIM_LOW_WATER=0.5  TP_CONTEXT_WINDOW=24576\n  TP_YIELD_SECONDS=900  TP_TOOL_STEPS=8  TP_KEEP_NOTES_ON_GAME_OVER=1\n  TP_BATCH_CAP=10\n\nFail-open: graft logic errors fall through to the stock method; the stock\nmethod is never wrapped in try/except, so its own errors propagate as stock.\n"""\nfrom __future__ import annotations\n\nimport os\nimport threading\nfrom typing import Any\n\n_tls = threading.local()\n_STATE = {"installed": False}\n# Pack 2 hook: called as ON_CUT(agent, dropped_messages) after a hysteresis cut.\nON_CUT = None\n\nDEFAULT_TRIM_LOW_WATER = 0.5\nDEFAULT_CONTEXT_WINDOW = 24576\nDEFAULT_YIELD_SECONDS = 900.0\nDEFAULT_TOOL_STEPS = 8\nDEFAULT_BATCH_CAP = 10\nBATCH_CAP_MESSAGE = (\n    "action batch cap reached: at most {cap} actions per python tool call. "\n    "Observe the results so far, then call the tool again for more actions."\n)\n_OFF = {"0", "false", "no", "off"}\n\n\n# ----------------------------------------------------------------- flags ---\ndef _env(name: str, default: str) -> str:\n    raw = os.environ.get(name)\n    return default if raw is None or not raw.strip() else raw.strip()\n\n\ndef enabled() -> bool:\n    return _env("TP_ENABLE", "1").lower() not in _OFF\n\n\ndef trim_low_water() -> float:\n    """Fraction of the budget to cut down to when over budget; 1.0 = stock."""\n    if not enabled():\n        return 1.0\n    try:\n        value = float(_env("TP_TRIM_LOW_WATER", str(DEFAULT_TRIM_LOW_WATER)))\n    except ValueError:\n        return DEFAULT_TRIM_LOW_WATER\n    return min(1.0, max(0.05, value))\n\n\ndef context_window() -> int:\n    """Context window for the request budget; 0 = leave stock."""\n    if not enabled():\n        return 0\n    try:\n        return max(0, int(_env("TP_CONTEXT_WINDOW", str(DEFAULT_CONTEXT_WINDOW))))\n    except ValueError:\n        return DEFAULT_CONTEXT_WINDOW\n\n\ndef yield_seconds() -> float:\n    """Turn yield in seconds; -1 = leave stock; 0 = disable the yield."""\n    if not enabled():\n        return -1.0\n    try:\n        return float(_env("TP_YIELD_SECONDS", str(DEFAULT_YIELD_SECONDS)))\n    except ValueError:\n        return DEFAULT_YIELD_SECONDS\n\n\ndef tool_steps() -> int:\n    """Calls per turn; -1 = leave stock; 0 = unlimited."""\n    if not enabled():\n        return -1\n    try:\n        return int(_env("TP_TOOL_STEPS", str(DEFAULT_TOOL_STEPS)))\n    except ValueError:\n        return DEFAULT_TOOL_STEPS\n\n\ndef keep_notes_on_game_over() -> bool:\n    if not enabled():\n        return False\n    return _env("TP_KEEP_NOTES_ON_GAME_OVER", "1").lower() not in _OFF\n\n\ndef batch_cap() -> int:\n    """Requested actions per python tool call; 0 = unlimited."""\n    if not enabled():\n        return 0\n    try:\n        return max(0, int(_env("TP_BATCH_CAP", str(DEFAULT_BATCH_CAP))))\n    except ValueError:\n        return DEFAULT_BATCH_CAP\n\n\n# ------------------------------------------------------------- seam: trim ---\ndef _patch_trim(cls: Any) -> None:\n    stock_trim = cls._trim_messages_for_context\n\n    def trim(self, messages, *, tools=None, preserve_recent=1, extra_safety_tokens=0):\n        low = trim_low_water()\n        if low >= 1.0 or not messages:\n            return stock_trim(self, messages, tools=tools, preserve_recent=preserve_recent,\n                              extra_safety_tokens=extra_safety_tokens)\n        try:\n            system_message = messages[0]\n            history = list(messages[1:])\n            preserve_recent = max(0, preserve_recent)\n            budget = max(1, self._context_budget_tokens - max(0, extra_safety_tokens))\n            estimate = self._estimate_request_input_tokens([system_message, *history], tools=tools)\n            if estimate <= budget:\n                return [system_message, *self._drop_until_first_user_message(history)]\n            target = max(1, int(budget * low))\n            original = list(history)\n\n            def user_count(items):\n                return sum(1 for m in items if str(m.get("role", "")).strip() == "user")\n\n            while history and estimate > target:\n                # Never cut away the most recent user message: a request whose\n                # history is only assistant/tool turns is rejected by the\n                # server ("No user query found in messages", measured live).\n                if user_count(history) <= 1:\n                    break\n                if not self._drop_oldest_history_block(history, preserve_recent=preserve_recent):\n                    break\n                estimate = self._estimate_request_input_tokens([system_message, *history], tools=tools)\n            history = self._drop_until_first_user_message(history)\n            if not history and original:\n                # fall back to the stock trim rather than send an empty history\n                return stock_trim(self, messages, tools=tools, preserve_recent=preserve_recent,\n                                  extra_safety_tokens=extra_safety_tokens)\n            dropped = original[: max(0, len(original) - len(history))]\n            hook = ON_CUT\n            if hook is not None and dropped:\n                try:\n                    hook(self, dropped)\n                except Exception:  # noqa: BLE001 — a summary failure never blocks the turn\n                    pass\n            return [system_message, *history]\n        except Exception:  # noqa: BLE001 — fail open to stock\n            return stock_trim(self, messages, tools=tools, preserve_recent=preserve_recent,\n                              extra_safety_tokens=extra_safety_tokens)\n\n    trim._tp_stock = stock_trim\n    cls._trim_messages_for_context = trim\n\n\n# ------------------------------------------------------------- seam: init ---\ndef _patch_init(cls: Any) -> None:\n    stock_init = cls.__init__\n\n    def init(self, *args, **kwargs):\n        stock_init(self, *args, **kwargs)\n        try:\n            window = context_window()\n            if window > 0:\n                self._context_budget_tokens = max(\n                    1024, window - self._reply_reserve_tokens - self._request_safety_margin_tokens)\n            ys = yield_seconds()\n            if ys >= 0:\n                self._yield_seconds = None if ys == 0 else float(ys)\n            ts = tool_steps()\n            if ts >= 0:\n                self._tool_steps = None if ts == 0 else max(1, ts)\n        except Exception:  # noqa: BLE001\n            pass\n\n    init._tp_stock = stock_init\n    cls.__init__ = init\n\n\n# ------------------------------------------------------------ seam: notes ---\ndef _patch_notes(cls: Any) -> None:\n    stock = cls._update_summarized_knowledge_from_step_summary\n\n    def update(self):\n        if not keep_notes_on_game_over():\n            return stock(self)\n        try:\n            summary = self._last_step_summary\n            if not summary:\n                return None\n            if summary.get("level_transition") or summary.get("run_complete"):\n                return stock(self)\n            return None  # game_over or nothing: keep every carried note\n        except Exception:  # noqa: BLE001\n            return stock(self)\n\n    update._tp_stock = stock\n    cls._update_summarized_knowledge_from_step_summary = update\n\n\n# -------------------------------------------------------- seam: batch cap ---\ndef begin_tool_call() -> None:\n    """Reset the per-tool-call action budget (called at every _run_python_tool)."""\n    _tls.remaining = batch_cap()\n\n\ndef _remaining() -> int | None:\n    cap = batch_cap()\n    if cap <= 0:\n        return None\n    remaining = getattr(_tls, "remaining", None)\n    if remaining is None:\n        remaining = cap\n        _tls.remaining = remaining\n    return remaining\n\n\ndef _patch_batch_cap(cls: Any) -> None:\n    stock_normalize = cls._normalize_python_actions\n    stock_run = cls._run_python_tool\n\n    def normalize(self, value):\n        normalized = stock_normalize(self, value)\n        try:\n            remaining = _remaining()\n        except Exception:  # noqa: BLE001\n            return normalized\n        if remaining is None:\n            return normalized\n        if remaining <= 0:\n            raise ValueError(BATCH_CAP_MESSAGE.format(cap=batch_cap()))\n        if len(normalized) > remaining:\n            normalized = normalized[:remaining]\n        _tls.remaining = remaining - len(normalized)\n        return normalized\n\n    def run(self, state_path, arguments):\n        begin_tool_call()\n        return stock_run(self, state_path, arguments)\n\n    normalize._tp_stock = stock_normalize\n    run._tp_stock = stock_run\n    cls._normalize_python_actions = normalize\n    cls._run_python_tool = run\n\n\n# ------------------------------------------------------------ time guard ---\ndef time_guard_per_game_s(stock_per_game_s: float, *, setup_elapsed_s: float, games: int,\n                          concurrency: int, total_budget_s: float = 32400.0,\n                          margin_s: float = 240.0) -> float:\n    """Shrink the per-game box only if setup + waves would overrun the 9 h box.\n\n    Never grows the box; never returns less than 600 s.\n    """\n    try:\n        waves = max(1, -(-int(games) // max(1, int(concurrency))))\n        fits = (float(total_budget_s) - float(setup_elapsed_s) - float(margin_s)) / waves\n        return float(min(float(stock_per_game_s), max(600.0, fits)))\n    except Exception:  # noqa: BLE001\n        return float(stock_per_game_s)\n\n\n# --------------------------------------------------------------- install ---\ndef install() -> str:\n    if _STATE["installed"]:\n        return "throughput: SKIP (already applied)"\n    try:\n        from inference.agent import tool_agent as agent_mod\n    except Exception as exc:  # noqa: BLE001\n        return f"throughput: SKIP (tool_agent module missing: {exc!r})"\n    cls = getattr(agent_mod, "ToolAgent", None)\n    if cls is None:\n        return "throughput: SKIP (missing ToolAgent)"\n    for name in ("_trim_messages_for_context", "_update_summarized_knowledge_from_step_summary",\n                 "_normalize_python_actions", "_run_python_tool", "_estimate_request_input_tokens",\n                 "_drop_oldest_history_block", "_drop_until_first_user_message"):\n        if getattr(cls, name, None) is None:\n            return f"throughput: SKIP (ToolAgent.{name} missing)"\n    _patch_trim(cls)\n    _patch_init(cls)\n    _patch_notes(cls)\n    _patch_batch_cap(cls)\n    _STATE["installed"] = True\n    return "throughput: OK"\n\n\ndef status() -> dict[str, Any]:\n    return {\n        "installed": _STATE["installed"],\n        "enabled": enabled(),\n        "trim_low_water": trim_low_water(),\n        "context_window": context_window(),\n        "yield_seconds": yield_seconds(),\n        "tool_steps": tool_steps(),\n        "keep_notes_on_game_over": keep_notes_on_game_over(),\n        "batch_cap": batch_cap(),\n    }\n',
    'graft_control.py': '"""Memory & control graft (Pack 2, 2026-08-29) — the loop keeps its state,\nprobes before it reasons, and notices when it is stuck.\n\nInstalled AFTER graft_throughput (it relies on its ON_CUT hook and wraps the\nseams it already wrapped). Every behaviour has a TP2_* flag read at call time;\nTP2_ENABLE=0 makes every seam a pass-through.\n\nSeams:\n  (A) SUMMARY  graft_throughput.ON_CUT + ToolAgent._update_summarized_knowledge_from_step_summary\n               harness-owned summary: when the trimmer cuts history, a short\n               non-thinking completion compresses the dropped turns into the\n               seven carried note fields (the existing prompt channel); on a\n               level-up the whole history is compressed into cross-level notes\n               + action model before the level-specific keys are wiped.\n  (B) PROBE    _HarnessGameSession.play — at game start each available\n               keyboard action is executed once and up to TP2_PROBE_CLICKS\n               salient components are clicked; the effect table rides the\n               user prompt while the agent is on that level.\n  (C) STALL    _HarnessGameSession._execute_action + ToolAgent._build_user_prompt\n               + ToolAgent.analyze — HUD-aware frame hashing; after\n               TP2_STALL_T1 actions without a new board state a STAGNATION\n               directive is appended to the prompt; after TP2_STALL_T2 (30) the\n               harness issues a level RESET (max TP2_STALL_RESETS_PER_LEVEL).\n  (D) STREAK   _HarnessGameSession.step_env — inside one python tool call,\n               after TP2_STREAK_N consecutive no-effect actions further\n               actions are refused with a readable error.\n  (E) DIFF     _HarnessGameSession._execute_action + ToolAgent._compact_action_result\n               every action result carries a compact diff summary (changed\n               cells, HUD-excluded count, bbox, colours added/removed).\n\nEvidence: docs/STRATEGY-2026-08-29-independent-review-path-to-6.md §4,\ndocs/research-2026-08-29/R2-literature-mechanisms.md, R4 §1.2/§4.\n"""\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nimport os\nimport threading\nimport time\nfrom collections import Counter, deque\nfrom typing import Any\n\n_STATE = {"installed": False}\n_OFF = {"0", "false", "no", "off"}\n_lock = threading.Lock()\n# stock ToolAgent.analyze as captured at install (tests may swap it)\n_ANALYZE_STOCK: dict[str, Any] = {}\n\nSUMMARY_SYSTEM_PROMPT = (\n    "You compress the working notes of an agent playing an unknown 64x64 grid game. "\n    "From the transcript, write EXACTLY these seven lines and nothing else:\\n"\n    "World model: <what the level contains and how it behaves, verified facts first>\\n"\n    "Goal model: <what seems to complete the level; mark guesses as guesses>\\n"\n    "Action model: <what each action does, with exact effects>\\n"\n    "Recent findings: <the newest confirmed observations>\\n"\n    "Open questions: <what is still unknown>\\n"\n    "Plan: <the best next steps>\\n"\n    "Cross-level notes: <rules likely to hold on later levels>\\n"\n    "Keep coordinates, colours and counts exact. Keep ruled-out hypotheses as ruled out. "\n    "At most 120 words total; one line per field, no blank lines."\n)\nLEVEL_SUMMARY_SYSTEM_PROMPT = (\n    "The agent just completed a level of an unknown 64x64 grid game and moves to the next "\n    "level of the same game. From the transcript and notes, write EXACTLY these two lines:\\n"\n    "Action model: <what each action does, with exact effects; controls usually persist>\\n"\n    "Cross-level notes: <mechanics, goal pattern and lessons that should transfer; "\n    "drop layout details specific to the finished level>\\n"\n    "At most 150 words total."\n)\nSTAGNATION_T1_TEXT = (\n    "STAGNATION WARNING: the last {n} actions produced NO board state you had not already "\n    "seen on this level (HUD counters ignored). Do not repeat that pattern. Enumerate every "\n    "(action, target) you have NOT tried on this level — untested keyboard actions, unclicked "\n    "objects, different orderings — and take the most informative untried one now."\n)\nSTAGNATION_RESET_TEXT = (\n    "HARNESS NOTE: the level was RESET by the harness at action {at} after {n} actions without "\n    "a new board state; the board is back at the level\'s opening state. Your earlier notes still "\n    "apply. Choose a different approach than the one that stalled."\n)\nSTREAK_MESSAGE = (\n    "no_effect_streak: the last {n} actions changed nothing on the board. The rest of this batch "\n    "was not executed. Observe current_frame and pick a different action or object."\n)\n\n\n# ----------------------------------------------------------------- flags ---\ndef _env(name: str, default: str) -> str:\n    raw = os.environ.get(name)\n    return default if raw is None or not raw.strip() else raw.strip()\n\n\ndef _flag(name: str, default: str = "1") -> bool:\n    return _env(name, default).lower() not in _OFF\n\n\ndef _int(name: str, default: int) -> int:\n    try:\n        return int(_env(name, str(default)))\n    except ValueError:\n        return default\n\n\ndef enabled() -> bool:\n    return _flag("TP2_ENABLE")\n\n\ndef summary_enabled() -> bool:\n    return enabled() and _flag("TP2_SUMMARY")\n\n\ndef probe_enabled() -> bool:\n    return enabled() and _flag("TP2_PROBE")\n\n\ndef stall_enabled() -> bool:\n    return enabled() and _flag("TP2_STALL")\n\n\ndef streak_enabled() -> bool:\n    return enabled() and _flag("TP2_STREAK")\n\n\ndef diff_enabled() -> bool:\n    return enabled() and _flag("TP2_DIFF")\n\n\ndef probe_clicks() -> int:\n    return max(0, _int("TP2_PROBE_CLICKS", 3))\n\n\ndef stall_t1() -> int:\n    return max(1, _int("TP2_STALL_T1", 10))\n\n\ndef stall_t2() -> int:\n    return max(stall_t1() + 1, _int("TP2_STALL_T2", 30))\n\n\ndef stall_resets_per_level() -> int:\n    return max(0, _int("TP2_STALL_RESETS_PER_LEVEL", 2))\n\n\ndef streak_n() -> int:\n    return max(1, _int("TP2_STREAK_N", 3))\n\n\ndef summary_max_tokens() -> int:\n    return max(100, _int("TP2_SUMMARY_MAX_TOKENS", 300))\n\n\ndef summary_min_interval_s() -> float:\n    try:\n        return max(0.0, float(_env("TP2_SUMMARY_MIN_INTERVAL_S", "240")))\n    except ValueError:\n        return 240.0\n\n\ndef summary_min_chars() -> int:\n    return max(0, _int("TP2_SUMMARY_MIN_CHARS", 2000))\n\n\ndef summary_async() -> bool:\n    return _flag("TP2_SUMMARY_ASYNC", "1")\n\n\ndef status() -> dict[str, Any]:\n    return {\n        "installed": _STATE["installed"], "enabled": enabled(),\n        "summary": summary_enabled(), "probe": probe_enabled(), "probe_clicks": probe_clicks(),\n        "stall": stall_enabled(), "stall_t1": stall_t1(), "stall_t2": stall_t2(),\n        "streak": streak_enabled(), "streak_n": streak_n(), "diff": diff_enabled(),\n    }\n\n\n# ------------------------------------------------------------ primitives ---\nGrid = tuple[tuple[int, ...], ...]\n\n\ndef grid_hash(grid: Grid, mask: set[tuple[int, int]] | None = None) -> str:\n    h = hashlib.sha1()\n    for r, row in enumerate(grid):\n        if mask:\n            row = tuple(0 if (r, c) in mask else v for c, v in enumerate(row))\n        h.update(bytes(int(v) & 0xFF for v in row))\n        h.update(b"\\n")\n    return h.hexdigest()[:16]\n\n\nclass HudMask:\n    """Online HUD/timer detector: cells that change in most transitions."""\n\n    def __init__(self, min_transitions: int = 8, threshold: float = 0.6) -> None:\n        self.counts: Counter = Counter()\n        self.n = 0\n        self.min_transitions = min_transitions\n        self.threshold = threshold\n        self.size: tuple[int, int] = (64, 64)\n\n    def observe(self, before: Grid, after: Grid) -> None:\n        if not before or not after or len(before) != len(after):\n            return\n        self.size = (len(after), len(after[0]) if after[0] else 0)\n        changed = 0\n        for r, (rb, ra) in enumerate(zip(before, after)):\n            if rb == ra:\n                continue\n            for c, (vb, va) in enumerate(zip(rb, ra)):\n                if vb != va:\n                    self.counts[(r, c)] += 1\n                    changed += 1\n        if changed:\n            self.n += 1\n\n    def mask(self, edge_band: int | None = None, band_threshold: float = 0.3,\n             size: tuple[int, int] | None = None) -> set[tuple[int, int]] | None:\n        """Volatile cells, extended to whole edge-band rows/columns (HUD bars\n        and multi-digit counters live on the border; only their fastest digit\n        clears the per-cell threshold, so mask the bar, not the digit)."""\n        if self.n < self.min_transitions:\n            return None\n        cut = self.threshold * self.n\n        out = {cell for cell, cnt in self.counts.items() if cnt > cut}\n        h, w = size or self.size\n        if edge_band is None:\n            edge_band = max(1, h // 16)\n        row_sum: Counter = Counter()\n        col_sum: Counter = Counter()\n        for (r, c), cnt in self.counts.items():\n            row_sum[r] += cnt\n            col_sum[c] += cnt\n        band_rows = [r for r in range(h) if r < edge_band or r >= h - edge_band]\n        band_cols = [c for c in range(w) if c < edge_band or c >= w - edge_band]\n        for r in band_rows:\n            if any((r, c) in out for c in range(w)) or row_sum[r] / (self.n * w) >= band_threshold:\n                out.update((r, c) for c in range(w))\n        for c in band_cols:\n            if any((r, c) in out for r in range(h)) or col_sum[c] / (self.n * h) >= band_threshold:\n                out.update((r, c) for r in range(h))\n        return out or None\n\n\ndef _edge_only(diff: dict[str, Any] | None, edge_band: int | None = None,\n               size: tuple[int, int] = (64, 64)) -> bool:\n    """True when every changed non-HUD cell lies inside the border band."""\n    if not diff or not diff.get("bbox") or diff.get("changed_ex_hud", 0) == 0:\n        return False\n    r0, c0, r1, c1 = diff["bbox"]\n    h, w = size\n    if edge_band is None:\n        edge_band = max(1, h // 16)\n    rows_in_band = (r1 < edge_band) or (r0 >= h - edge_band)\n    cols_in_band = (c1 < edge_band) or (c0 >= w - edge_band)\n    return rows_in_band or cols_in_band\n\n\ndef diff_summary(before: Grid, after: Grid, mask: set[tuple[int, int]] | None = None) -> dict[str, Any]:\n    cells: list[tuple[int, int]] = []\n    added: Counter = Counter()\n    removed: Counter = Counter()\n    if before and after and len(before) == len(after):\n        for r, (rb, ra) in enumerate(zip(before, after)):\n            if rb == ra:\n                continue\n            for c, (vb, va) in enumerate(zip(rb, ra)):\n                if vb != va:\n                    cells.append((r, c))\n                    added[int(va)] += 1\n                    removed[int(vb)] += 1\n    ex = [cell for cell in cells if not mask or cell not in mask]\n    box = ex or cells\n    bbox = [min(r for r, _ in box), min(c for _, c in box), max(r for r, _ in box), max(c for _, c in box)] if box else None\n    return {\n        "changed": len(cells),\n        "changed_ex_hud": len(ex),\n        "bbox": bbox,\n        "colors_added": sorted(added),\n        "colors_removed": sorted(removed),\n    }\n\n\ndef components(grid: Grid, background: int | None = None) -> list[dict[str, Any]]:\n    if not grid:\n        return []\n    h, w = len(grid), len(grid[0])\n    if background is None:\n        background = Counter(v for row in grid for v in row).most_common(1)[0][0]\n    seen = [[False] * w for _ in range(h)]\n    out: list[dict[str, Any]] = []\n    for r0 in range(h):\n        for c0 in range(w):\n            if seen[r0][c0] or grid[r0][c0] == background:\n                continue\n            color = grid[r0][c0]\n            stack = [(r0, c0)]\n            seen[r0][c0] = True\n            cells = []\n            while stack:\n                r, c = stack.pop()\n                cells.append((r, c))\n                for nr, nc in ((r - 1, c), (r + 1, c), (r, c - 1), (r, c + 1)):\n                    if 0 <= nr < h and 0 <= nc < w and not seen[nr][nc] and grid[nr][nc] == color:\n                        seen[nr][nc] = True\n                        stack.append((nr, nc))\n            rs = [r for r, _ in cells]\n            cs = [c for _, c in cells]\n            out.append({\n                "color": int(color), "area": len(cells), "cells": cells,\n                "bbox": [min(rs), min(cs), max(rs), max(cs)],\n                "center": (round(sum(rs) / len(rs)), round(sum(cs) / len(cs))),\n            })\n    out.sort(key=lambda d: d["area"])\n    return out\n\n\ndef salient_clicks(grid: Grid, k: int) -> list[tuple[int, int, dict[str, Any]]]:\n    """Centres of up to k small, rare-colour, non-background components."""\n    comps = components(grid)\n    if not comps or k <= 0:\n        return []\n    color_area: Counter = Counter()\n    for comp in comps:\n        color_area[comp["color"]] += comp["area"]\n\n    def score(comp: dict[str, Any]) -> tuple:\n        small = 0 if 2 <= comp["area"] <= 60 else 1\n        return (small, color_area[comp["color"]], comp["area"])\n\n    picked: list[tuple[int, int, dict[str, Any]]] = []\n    used_colors: Counter = Counter()\n    for comp in sorted(comps, key=score):\n        if used_colors[comp["color"]] >= 2:\n            continue\n        r, c = comp["center"]\n        if (r, c) not in set(comp["cells"]):\n            r, c = min(comp["cells"], key=lambda rc: abs(rc[0] - r) + abs(rc[1] - c))\n        picked.append((r, c, {"color": comp["color"], "area": comp["area"], "bbox": comp["bbox"]}))\n        used_colors[comp["color"]] += 1\n        if len(picked) >= k:\n            break\n    return picked\n\n\n# ------------------------------------------------------- session state ---\nclass SessionState:\n    def __init__(self) -> None:\n        self.hud = HudMask()\n        self.seen: set[str] = set()\n        self.since_new = 0\n        self.level: int | None = None\n        self.resets_this_level = 0\n        self.last_reset_note: str | None = None\n        self.streak = 0\n        self.actions = 0\n        self.recent_hashes: deque = deque(maxlen=64)\n\n\ndef _state(session: Any) -> SessionState:\n    st = getattr(session, "_tp2", None)\n    if st is None:\n        st = SessionState()\n        try:\n            session._tp2 = st\n        except Exception:  # noqa: BLE001\n            pass\n    return st\n\n\ndef _reset_available(session: Any, arcengine: Any) -> bool:\n    """RESET is filtered out of the model-facing valid_actions by the solver;\n    ask the engine state directly (absent list => assume available)."""\n    try:\n        available = session.game.current_state.available_actions\n    except Exception:  # noqa: BLE001\n        return True\n    try:\n        ids = set(int(a) for a in (available or []))\n    except Exception:  # noqa: BLE001\n        return True\n    return (not ids) or int(arcengine.GameAction.RESET.value) in ids\n\n\ndef _session_of(agent: Any) -> Any:\n    cb = getattr(agent, "_step_env_callback", None)\n    return getattr(cb, "__self__", None)\n\n\ndef _grid(session: Any, solver_mod: Any) -> Grid:\n    try:\n        return solver_mod._grid_from_state(session.game.current_state)\n    except Exception:  # noqa: BLE001\n        return ()\n\n\ndef _after_action(st: SessionState, before: Grid, after: Grid, payload: dict[str, Any]) -> None:\n    """Update diff / HUD / stall / streak state after one executed action."""\n    mask = st.hud.mask()\n    diff = diff_summary(before, after, mask) if (before and after) else None\n    if diff is not None and diff_enabled():\n        payload["diff"] = diff\n    st.hud.observe(before, after)\n    st.actions += 1\n    level = payload.get("level")\n    if level is not None and level != st.level:\n        st.level = level\n        st.seen = set()\n        st.since_new = 0\n        st.resets_this_level = 0\n    if after:\n        h = grid_hash(after, mask)\n        st.recent_hashes.append(h)\n        if h in st.seen:\n            st.since_new += 1\n        elif diff is not None and (diff["changed_ex_hud"] == 0\n                                   or _edge_only(diff, size=(len(after), len(after[0]) if after[0] else 0))):\n            # only HUD/border cells moved: not a new gameplay state\n            st.seen.add(h)\n            st.since_new += 1\n        else:\n            st.seen.add(h)\n            st.since_new = 0\n    if payload.get("executed"):\n        no_effect = (diff["changed_ex_hud"] == 0) if diff is not None else (not payload.get("board_changed"))\n        animated = int(payload.get("frame_count") or 1) > 1\n        st.streak = st.streak + 1 if (no_effect and not animated) else 0\n\n\n# ------------------------------------------------------------ summaries ---\ndef _transcript(messages: list[dict[str, Any]], cap: int = 14000) -> str:\n    parts: list[str] = []\n    for m in messages:\n        role = str(m.get("role", ""))\n        if role == "assistant":\n            content = m.get("content")\n            if isinstance(content, str) and content.strip():\n                parts.append("ASSISTANT: " + content.strip()[:1500])\n            for call in m.get("tool_calls") or []:\n                fn = call.get("function", {}) if isinstance(call, dict) else {}\n                args = fn.get("arguments", "")\n                if isinstance(args, dict):\n                    args = json.dumps(args)\n                parts.append("TOOL CALL: " + str(args)[:600])\n        elif role == "tool":\n            parts.append("TOOL RESULT: " + str(m.get("content", ""))[:700])\n        elif role == "user":\n            content = m.get("content")\n            if isinstance(content, list):\n                content = " ".join(str(p.get("text", "")) for p in content if isinstance(p, dict))\n            text = str(content or "")\n            head = text.split("\\n", 2)[:2]\n            parts.append("USER: " + " ".join(head)[:300])\n    text = "\\n".join(parts)\n    return text[-cap:] if len(text) > cap else text\n\n\ndef _post_summary(agent: Any, system_prompt: str, user_text: str, agent_mod: Any) -> str:\n    import requests  # noqa: PLC0415 — the bundle already depends on it\n\n    from inference.utils.openai_compat import build_chat_payload  # noqa: PLC0415\n\n    model = agent._model\n    payload = build_chat_payload(\n        provider=model.provider, model=model.model_id,\n        messages=[{"role": "system", "content": system_prompt}, {"role": "user", "content": user_text}],\n        max_tokens=summary_max_tokens(), temperature=0.2, top_p=0.95, top_k=20,\n        thinking=False, tools=None, tool_choice=None, seed=None,\n    )\n    response = requests.post(f"{model.base_url.rstrip(\'/\')}/chat/completions",\n                             headers=agent._headers(), json=payload, timeout=180)\n    response.raise_for_status()\n    message = response.json()["choices"][0]["message"]\n    content = message.get("content") or ""\n    if isinstance(content, list):\n        content = " ".join(str(p.get("text", "")) for p in content if isinstance(p, dict))\n    return str(content)\n\n\ndef _notes_text(agent: Any) -> str:\n    notes = getattr(agent, "_summarized_knowledge", {}) or {}\n    labels = [("World model", "world_model"), ("Goal model", "goal_model"), ("Action model", "action_model"),\n              ("Recent findings", "recent_findings"), ("Open questions", "open_questions"),\n              ("Plan", "current_plan"), ("Cross-level notes", "cross_level_notes")]\n    lines = [f"{label}: {notes.get(key)}" for label, key in labels if notes.get(key)]\n    return "\\n".join(lines)\n\n\ndef _summary_allowed(agent: Any, text: str) -> bool:\n    """Rate limit: skip tiny cuts and cuts closer than the minimum interval."""\n    if len(text) < summary_min_chars():\n        return False\n    now = time.monotonic()\n    last = getattr(agent, "_tp2_last_summary_at", None)\n    if last is not None and (now - last) < summary_min_interval_s():\n        return False\n    try:\n        agent._tp2_last_summary_at = now\n    except Exception:  # noqa: BLE001\n        pass\n    return True\n\n\ndef _summarize_into_notes(agent: Any, dropped: list[dict[str, Any]], agent_mod: Any,\n                          *, force: bool = False) -> bool:\n    text = _transcript(dropped)\n    if not text.strip():\n        return False\n    if not force and not _summary_allowed(agent, text):\n        return False\n    prev = _notes_text(agent)\n    user_text = ("Previous notes:\\n" + prev + "\\n\\n" if prev else "") + "Transcript of the turns being compressed:\\n" + text\n    content = _post_summary(agent, SUMMARY_SYSTEM_PROMPT, user_text, agent_mod)\n    note = agent_mod._extract_scientist_note(content)\n    if not note or not any(note.values()):\n        return False\n    with _lock:\n        for key, value in note.items():\n            if value:\n                agent._summarized_knowledge[key] = value\n    return True\n\n\ndef _summarize_level_boundary(agent: Any, agent_mod: Any) -> dict[str, str]:\n    history = list(getattr(agent, "_history_messages", []) or [])\n    text = _transcript(history)\n    prev = _notes_text(agent)\n    user_text = ("Notes so far:\\n" + prev + "\\n\\n" if prev else "") + "Transcript:\\n" + text\n    content = _post_summary(agent, LEVEL_SUMMARY_SYSTEM_PROMPT, user_text, agent_mod)\n    note = agent_mod._extract_scientist_note(content)\n    return {k: v for k, v in (note or {}).items() if v and k in ("action_model", "cross_level_notes")}\n\n\n# --------------------------------------------------------------- probe ---\n_KEYBOARD = [("ACTION1", "UP"), ("ACTION2", "DOWN"), ("ACTION3", "LEFT"), ("ACTION4", "RIGHT"), ("ACTION5", "SPACE")]\n\n\ndef _fmt_effect(diff: dict[str, Any] | None, payload: dict[str, Any]) -> str:\n    if payload.get("level_completed"):\n        return "COMPLETED THE LEVEL"\n    if payload.get("game_over"):\n        return "GAME OVER (level was reset)"\n    if not diff:\n        return "board changed" if payload.get("board_changed") else "no effect"\n    if diff["changed_ex_hud"] == 0 and diff["changed"] == 0:\n        return "no effect"\n    if diff["changed_ex_hud"] == 0:\n        return f"only HUD-like cells changed ({diff[\'changed\']})"\n    b = diff["bbox"]\n    return (f"changed {diff[\'changed_ex_hud\']} cells in rows {b[0]}-{b[2]} cols {b[1]}-{b[3]}; "\n            f"colours appeared {diff[\'colors_added\']} vanished {diff[\'colors_removed\']}")\n\n\ndef run_probe(session: Any, solver_mod: Any, arcengine: Any) -> dict[str, Any] | None:\n    """Execute the level-start probe on a fresh game. Returns the table record."""\n    game = session.game\n    state = game.current_state\n    try:\n        available = set(int(a) for a in state.available_actions)\n    except Exception:  # noqa: BLE001\n        return None\n    st = _state(session)\n    try:\n        level = int(solver_mod._level_number(game))\n    except Exception:  # noqa: BLE001\n        level = int(getattr(state, "levels_completed", 0) or 0) + 1\n    lines: list[str] = []\n    executed = 0\n\n    def do(action_name: str, data: dict[str, Any], label: str) -> dict[str, Any]:\n        nonlocal executed\n        action = arcengine.ActionInput(id=arcengine.GameAction.from_name(action_name), data=data)\n        before = _grid(session, solver_mod)\n        payload = session._execute_action(action, batch_index=1, batch_size=1, generated_tokens=0)\n        executed += 1\n        diff = payload.get("diff")\n        if diff is None:\n            after = _grid(session, solver_mod)\n            diff = diff_summary(before, after, st.hud.mask()) if before and after else None\n        lines.append(f"- {label}: {_fmt_effect(diff, payload)}")\n        return payload\n\n    stop = False\n    for engine_name, label in _KEYBOARD:\n        value = int(arcengine.GameAction.from_name(engine_name).value)\n        if value not in available:\n            continue\n        payload = do(engine_name, {}, label)\n        if payload.get("level_completed") or payload.get("run_complete") or payload.get("game_over"):\n            stop = True\n            break\n        try:\n            available = set(int(a) for a in game.current_state.available_actions)\n        except Exception:  # noqa: BLE001\n            pass\n    click_value = int(arcengine.GameAction.from_name("ACTION6").value)\n    if not stop and click_value in available and probe_clicks() > 0:\n        grid = _grid(session, solver_mod)\n        for r, c, info in salient_clicks(grid, probe_clicks()):\n            payload = do("ACTION6", {"x": int(c), "y": int(r)},\n                         f"MOUSE(row {r}, col {c}) on a colour-{info[\'color\']} object of {info[\'area\']} cells")\n            if payload.get("level_completed") or payload.get("run_complete") or payload.get("game_over"):\n                break\n    if not lines:\n        return None\n    text = ("Harness probe at the start of this level (each available action tried once; "\n            f"{executed} actions spent, all recorded in `history`):\\n" + "\\n".join(lines))\n    return {"level": level, "text": text, "actions": executed}\n\n\n# --------------------------------------------------------------- install ---\ndef install() -> str:\n    if _STATE["installed"]:\n        return "control: SKIP (already applied)"\n    try:\n        import graft_throughput as tp  # noqa: PLC0415\n    except Exception as exc:  # noqa: BLE001\n        return f"control: SKIP (graft_throughput missing: {exc!r})"\n    if not tp._STATE.get("installed"):\n        return "control: SKIP (graft_throughput not installed)"\n    try:\n        import arcengine  # noqa: PLC0415\n        from inference.agent import tool_agent as agent_mod  # noqa: PLC0415\n        from inference.framework import solver as solver_mod  # noqa: PLC0415\n    except Exception as exc:  # noqa: BLE001\n        return f"control: SKIP (import failed: {exc!r})"\n    agent_cls = agent_mod.ToolAgent\n    session_cls = solver_mod._HarnessGameSession\n    for name in ("_build_user_prompt", "analyze", "_compact_action_result", "_run_python_tool",\n                 "_update_summarized_knowledge_from_step_summary"):\n        if getattr(agent_cls, name, None) is None:\n            return f"control: SKIP (ToolAgent.{name} missing)"\n    for name in ("play", "step_env", "_execute_action", "_error_payload"):\n        if getattr(session_cls, name, None) is None:\n            return f"control: SKIP (_HarnessGameSession.{name} missing)"\n\n    # (E)+(C)+(D) state: _execute_action ---------------------------------\n    stock_execute_action = session_cls._execute_action\n\n    def execute_action_wrapped(self, action, *args, **kwargs):\n        before = _grid(self, solver_mod) if enabled() else ()\n        payload = stock_execute_action(self, action, *args, **kwargs)\n        if not enabled():\n            return payload\n        try:\n            _after_action(_state(self), before, _grid(self, solver_mod), payload)\n        except Exception:  # noqa: BLE001\n            pass\n        return payload\n\n    execute_action_wrapped._tp2_stock = stock_execute_action\n    session_cls._execute_action = execute_action_wrapped\n\n    stock_aggregate = agent_mod._aggregate_action_batch_result\n\n    def aggregate(*args, **kwargs):\n        out = stock_aggregate(*args, **kwargs)\n        try:\n            if diff_enabled():\n                executed = kwargs.get("executed_results") if "executed_results" in kwargs else (args[1] if len(args) > 1 else [])\n                diffs = [r.get("diff") for r in (executed or []) if isinstance(r, dict) and isinstance(r.get("diff"), dict)]\n                if diffs:\n                    last = dict(diffs[-1])\n                    last["batch_changed_ex_hud_total"] = sum(int(d.get("changed_ex_hud", 0)) for d in diffs)\n                    out["diff"] = last\n        except Exception:  # noqa: BLE001\n            pass\n        return out\n\n    aggregate._tp2_stock = stock_aggregate\n    agent_mod._aggregate_action_batch_result = aggregate\n\n    stock_compact = agent_cls._compact_action_result\n\n    def compact(self, payload):\n        out = stock_compact(self, payload)\n        try:\n            if diff_enabled() and isinstance(payload, dict) and isinstance(payload.get("diff"), dict):\n                out["diff"] = dict(payload["diff"])\n        except Exception:  # noqa: BLE001\n            pass\n        return out\n\n    compact._tp2_stock = stock_compact\n    agent_cls._compact_action_result = compact\n\n    # (D) streak halt: step_env + reset at each python tool call ----------\n    stock_step = session_cls.step_env\n\n    def step_env(self, arguments):\n        try:\n            if streak_enabled() and not (isinstance(arguments, dict) and arguments.get("query")):\n                st = _state(self)\n                if st.streak >= streak_n():\n                    return self._error_payload(STREAK_MESSAGE.format(n=st.streak))\n        except Exception:  # noqa: BLE001\n            pass\n        return stock_step(self, arguments)\n\n    step_env._tp2_stock = stock_step\n    session_cls.step_env = step_env\n\n    stock_run = agent_cls._run_python_tool\n\n    def run_tool(self, state_path, arguments):\n        try:\n            sess = _session_of(self)\n            if sess is not None:\n                _state(sess).streak = 0\n        except Exception:  # noqa: BLE001\n            pass\n        return stock_run(self, state_path, arguments)\n\n    run_tool._tp2_stock = stock_run\n    agent_cls._run_python_tool = run_tool\n\n    # (B)+(C) prompt: probe table + stagnation directive -----------------\n    stock_prompt = agent_cls._build_user_prompt\n\n    def build_prompt(self, action_num, **kwargs):\n        text = stock_prompt(self, action_num, **kwargs)\n        if not enabled():\n            return text\n        try:\n            extra: list[str] = []\n            current_frame = kwargs.get("current_frame")\n            level = getattr(current_frame, "level", None)\n            probe = getattr(self, "_tp2_probe", None)\n            if probe and probe_enabled() and (level is None or int(level) == int(probe["level"])):\n                extra.append(probe["text"])\n            sess = _session_of(self)\n            if sess is not None and stall_enabled():\n                st = _state(sess)\n                if st.last_reset_note:\n                    extra.append(st.last_reset_note)\n                    st.last_reset_note = None\n                if st.since_new >= stall_t1():\n                    extra.append(STAGNATION_T1_TEXT.format(n=st.since_new))\n            if extra:\n                text = text + "\\n" + "\\n".join(extra)\n        except Exception:  # noqa: BLE001\n            pass\n        return text\n\n    build_prompt._tp2_stock = stock_prompt\n    agent_cls._build_user_prompt = build_prompt\n\n    stock_analyze = agent_cls.analyze\n\n    def analyze(self, state_path, action_num, valid_actions=None, step_env=None, **kwargs):\n        try:\n            sess = getattr(step_env, "__self__", None)\n            if sess is not None and stall_enabled():\n                st = _state(sess)\n                if st.since_new >= stall_t2() and st.resets_this_level < stall_resets_per_level() \\\n                        and _reset_available(sess, arcengine):\n                    action = arcengine.ActionInput(id=arcengine.GameAction.RESET, data={})\n                    sess._execute_action(action, batch_index=1, batch_size=1, generated_tokens=0)\n                    st.resets_this_level += 1\n                    st.last_reset_note = STAGNATION_RESET_TEXT.format(at=sess.action_count, n=st.since_new)\n                    st.since_new = 0\n                    try:\n                        sess.write_runtime_state()\n                    except Exception:  # noqa: BLE001\n                        pass\n                    action_num = sess.action_count\n                    valid_actions = solver_mod._engine_action_names(sess.game)\n        except Exception:  # noqa: BLE001\n            pass\n        return _ANALYZE_STOCK["fn"](self, state_path, action_num, valid_actions=valid_actions,\n                                    step_env=step_env, **kwargs)\n\n    _ANALYZE_STOCK["fn"] = stock_analyze\n    analyze._tp2_stock = stock_analyze\n    agent_cls.analyze = analyze\n\n    # (B) probe at game start ---------------------------------------------\n    stock_play = session_cls.play\n\n    def play(self):\n        if probe_enabled():\n            try:\n                if int(getattr(self, "action_count", 0) or 0) == 0:\n                    self.seed_initial_history()\n                    record = run_probe(self, solver_mod, arcengine)\n                    if record:\n                        self.analyzer._tp2_probe = record\n                        self.write_runtime_state()\n            except Exception:  # noqa: BLE001\n                pass\n        return stock_play(self)\n\n    play._tp2_stock = stock_play\n    session_cls.play = play\n\n    # (A) summaries -------------------------------------------------------\n    def on_cut(agent, dropped):\n        if not summary_enabled():\n            return\n        if not summary_async():\n            _summarize_into_notes(agent, dropped, agent_mod)\n            return\n        # Rate-limit on the caller\'s thread, then run the summary call in the\n        # background so the turn is never blocked; the merge lands under a\n        # lock and the next prompt build picks it up (measured: a blocking\n        # 600-token summary cost ~70 s per cut at concurrency 28).\n        text = _transcript(list(dropped))\n        if not _summary_allowed(agent, text):\n            return\n        if getattr(agent, "_tp2_summary_inflight", False):\n            return\n        agent._tp2_summary_inflight = True\n        snapshot = list(dropped)\n\n        def worker():\n            try:\n                _summarize_into_notes(agent, snapshot, agent_mod, force=True)\n            except Exception:  # noqa: BLE001\n                pass\n            finally:\n                agent._tp2_summary_inflight = False\n\n        threading.Thread(target=worker, name="tp2-summary", daemon=True).start()\n\n    tp.ON_CUT = on_cut\n\n    stock_notes = agent_cls._update_summarized_knowledge_from_step_summary\n\n    def update_notes(self):\n        try:\n            summary = self._last_step_summary\n            if summary_enabled() and summary and summary.get("level_transition"):\n                carried = _summarize_level_boundary(self, agent_mod)\n                result = stock_notes(self)\n                for key, value in carried.items():\n                    self._summarized_knowledge[key] = value\n                return result\n        except Exception:  # noqa: BLE001\n            pass\n        return stock_notes(self)\n\n    update_notes._tp2_stock = stock_notes\n    agent_cls._update_summarized_knowledge_from_step_summary = update_notes\n\n    _STATE["installed"] = True\n    return "control: OK"\n',
    'frontier_explorer.py': '"""Frontier explorer — a model-free level explorer (Pack 4, 2026-08-29).\n\nA compact port of the "just-explore" method (dolphin-in-a-coma/arc-agi-3-\njust-explore, MIT; 3rd in the ARC-AGI-3 preview with zero LLM calls; median\n17 private levels after the graph-reset fix). The idea:\n\n  * Frame processing: 4-connected single-colour segments; status bars are\n    segments hugging a border (aspect ratio >= 5, or >= 3 same-shaped twins on\n    the same border) and are masked before hashing; click candidates are one\n    per segment, grouped into five priority tiers (salient colour + medium\n    size first, status-bar segments last); keyboard actions sit in tier 0.\n  * Level graph: nodes are masked-frame hashes; each node keeps its untested\n    candidates; the explorer takes an untested candidate in the active tier\n    at the current node, or walks the shortest path to the nearest node that\n    still has one (frontier); when nothing is reachable it opens the next tier.\n\nPure Python, no numpy; a 64x64 frame segments in a few milliseconds.\n"""\nfrom __future__ import annotations\n\nimport hashlib\nimport random\nfrom collections import Counter, deque\nfrom dataclasses import dataclass, field\nfrom typing import Any\n\nGrid = tuple[tuple[int, ...], ...]\n\nSALIENT = set(range(6, 16))\nMIN_WIDTH, MAX_WIDTH = 2, 32\nEDGE_DIST = 3\nBAR_RATIO = 5\nTWINS = 3\nN_GROUPS = 5\nKEYBOARD = {1: "ACTION1", 2: "ACTION2", 3: "ACTION3", 4: "ACTION4", 5: "ACTION5"}\n\n\n# ---------------------------------------------------------------- frames ---\ndef segments(grid: Grid) -> list[dict[str, Any]]:\n    """4-connected same-colour components over the whole grid (background too)."""\n    if not grid:\n        return []\n    h, w = len(grid), len(grid[0])\n    label = [[-1] * w for _ in range(h)]\n    out: list[dict[str, Any]] = []\n    for r0 in range(h):\n        for c0 in range(w):\n            if label[r0][c0] >= 0:\n                continue\n            color = grid[r0][c0]\n            idx = len(out)\n            label[r0][c0] = idx\n            stack = [(r0, c0)]\n            cells = []\n            while stack:\n                r, c = stack.pop()\n                cells.append((r, c))\n                for nr, nc in ((r - 1, c), (r + 1, c), (r, c - 1), (r, c + 1)):\n                    if 0 <= nr < h and 0 <= nc < w and label[nr][nc] < 0 and grid[nr][nc] == color:\n                        label[nr][nc] = idx\n                        stack.append((nr, nc))\n            rs = [r for r, _ in cells]\n            cs = [c for _, c in cells]\n            r_min, r_max, c_min, c_max = min(rs), max(rs), min(cs), max(cs)\n            out.append({\n                "id": idx, "color": int(color), "area": len(cells), "cells": cells,\n                "bbox": (r_min, c_min, r_max, c_max),\n                "height": r_max - r_min + 1, "width": c_max - c_min + 1,\n                "rect": len(cells) == (r_max - r_min + 1) * (c_max - c_min + 1),\n            })\n    return out\n\n\ndef _edges_of(seg: dict[str, Any], h: int, w: int) -> list[str]:\n    r_min, c_min, r_max, c_max = seg["bbox"]\n    edges = []\n    if c_max < EDGE_DIST:\n        edges.append("left")\n    if c_min > w - 1 - EDGE_DIST:\n        edges.append("right")\n    if r_max < EDGE_DIST:\n        edges.append("top")\n    if r_min > h - 1 - EDGE_DIST:\n        edges.append("bottom")\n    return edges\n\n\ndef status_bar_mask(grid: Grid, segs: list[dict[str, Any]]) -> set[tuple[int, int]]:\n    """just-explore\'s rule: border-hugging bars (aspect >= 5) or >= 3 twins on a border."""\n    if not grid:\n        return set()\n    h, w = len(grid), len(grid[0])\n    mask: set[tuple[int, int]] = set()\n    by_edge: dict[str, list[dict[str, Any]]] = {}\n    for seg in segs:\n        for e in _edges_of(seg, h, w):\n            by_edge.setdefault(e, []).append(seg)\n    for edge, group in by_edge.items():\n        horizontal = edge in ("top", "bottom")\n        for seg in group:\n            ratio = seg["width"] / seg["height"] if seg["height"] else 0\n            is_bar = (ratio >= BAR_RATIO) if horizontal else (ratio <= 1 / BAR_RATIO if ratio else False)\n            twins = [t for t in group if t is not seg and t["color"] == seg["color"]\n                     and t["rect"] == seg["rect"] and t["width"] == seg["width"] and t["height"] == seg["height"]]\n            if is_bar or len(twins) + 1 >= TWINS:\n                mask.update(seg["cells"])\n    return mask\n\n\ndef frame_hash(grid: Grid, mask: set[tuple[int, int]]) -> str:\n    h = hashlib.sha1()\n    for r, row in enumerate(grid):\n        vals = bytes((16 if (r, c) in mask else int(v)) & 0xFF for c, v in enumerate(row))\n        h.update(vals)\n        h.update(b"\\n")\n    return h.hexdigest()[:20]\n\n\ndef group_of(seg: dict[str, Any], masked: bool) -> int:\n    salient = seg["color"] in SALIENT\n    medium = MIN_WIDTH <= seg["width"] <= MAX_WIDTH and MIN_WIDTH <= seg["height"] <= MAX_WIDTH\n    if masked:\n        return 4\n    if salient and medium:\n        return 0\n    if medium:\n        return 1\n    if salient:\n        return 2\n    return 3\n\n\n@dataclass\nclass Candidate:\n    kind: str                 # "key" or "click"\n    action: str               # engine action name\n    data: dict[str, Any]      # {} or {"x":..,"y":..}\n    group: int\n    label: str\n\n\ndef candidates_for(grid: Grid, available: set[int], mask: set[tuple[int, int]],\n                   segs: list[dict[str, Any]], rng: random.Random) -> list[Candidate]:\n    out: list[Candidate] = []\n    for value, name in KEYBOARD.items():\n        if value in available:\n            out.append(Candidate("key", name, {}, 0, name))\n    if 6 in available:\n        for seg in segs:\n            cells = seg["cells"]\n            masked = all(cell in mask for cell in cells[: min(len(cells), 8)])\n            r, c = cells[rng.randrange(len(cells))]\n            out.append(Candidate("click", "ACTION6", {"x": int(c), "y": int(r)}, group_of(seg, masked),\n                                 f"click({r},{c}) colour {seg[\'color\']} area {seg[\'area\']}"))\n    return out\n\n\n# ----------------------------------------------------------------- graph ---\n@dataclass\nclass Node:\n    key: str\n    candidates: list[Candidate]\n    untested: dict[int, set[int]] = field(default_factory=dict)   # group -> candidate indices\n    result: dict[int, int] = field(default_factory=dict)          # idx -> 1 changed / -1 no change\n    target: dict[int, str] = field(default_factory=dict)          # idx -> node key\n\n    def __post_init__(self) -> None:\n        for i, cand in enumerate(self.candidates):\n            self.untested.setdefault(cand.group, set()).add(i)\n\n    def open_in(self, active: int) -> list[int]:\n        return [i for g in range(active + 1) for i in sorted(self.untested.get(g, ()))]\n\n\nclass FrontierExplorer:\n    def __init__(self, seed: int = 0) -> None:\n        self.rng = random.Random(seed)\n        self.reset()\n\n    def reset(self) -> None:\n        self.nodes: dict[str, Node] = {}\n        self.rev: dict[str, set[tuple[str, int]]] = {}     # target -> {(source, idx)}\n        self.active = 0\n        self.dist: dict[str, int] = {}\n        self.next_hop: dict[str, int] = {}                 # node -> candidate idx towards frontier\n        self.last: tuple[str, int] | None = None\n        self.mask: set[tuple[int, int]] | None = None\n        self.stats = Counter()\n\n    # -- observation --------------------------------------------------------\n    def observe(self, grid: Grid, available: list[int] | set[int]) -> str:\n        segs = segments(grid)\n        if self.mask is None:\n            self.mask = status_bar_mask(grid, segs)\n        key = frame_hash(grid, self.mask)\n        if key not in self.nodes:\n            cands = candidates_for(grid, set(int(a) for a in available), self.mask, segs, self.rng)\n            self.nodes[key] = Node(key, cands)\n            self.stats["nodes"] += 1\n            self._rebuild()\n        return key\n\n    # -- learning -----------------------------------------------------------\n    def record(self, prev_key: str, idx: int, new_key: str) -> None:\n        node = self.nodes.get(prev_key)\n        if node is None or idx not in range(len(node.candidates)):\n            return\n        changed = new_key != prev_key\n        node.result[idx] = 1 if changed else -1\n        node.untested.get(node.candidates[idx].group, set()).discard(idx)\n        if changed:\n            node.target[idx] = new_key\n            self.rev.setdefault(new_key, set()).add((prev_key, idx))\n            self.stats["edges"] += 1\n        else:\n            self.stats["noops"] += 1\n        self._rebuild()\n\n    # -- decision -----------------------------------------------------------\n    def choose(self, key: str) -> tuple[int, str]:\n        node = self.nodes[key]\n        while True:\n            open_idx = node.open_in(self.active)\n            if open_idx:\n                # lowest group first, random inside the group\n                best_group = min(node.candidates[i].group for i in open_idx)\n                pool = [i for i in open_idx if node.candidates[i].group == best_group]\n                i = self.rng.choice(pool)\n                self.stats["explore"] += 1\n                return i, f"untested tier {best_group}"\n            hop = self.next_hop.get(key)\n            if hop is not None:\n                self.stats["travel"] += 1\n                return hop, f"towards frontier (dist {self.dist.get(key)})"\n            if self.active < N_GROUPS - 1:\n                self.active += 1\n                self.stats["tier_advance"] += 1\n                self._rebuild()\n                continue\n            # everything exhausted: random tested-changing edge, else random candidate\n            changing = [i for i, r in node.result.items() if r == 1]\n            i = self.rng.choice(changing) if changing else self.rng.randrange(len(node.candidates))\n            self.stats["random"] += 1\n            return i, "exhausted: random"\n\n    # -- distances (BFS from frontier over reverse edges) --------------------\n    def _rebuild(self) -> None:\n        frontier = [k for k, n in self.nodes.items() if n.open_in(self.active)]\n        dist: dict[str, int] = {k: 0 for k in frontier}\n        hop: dict[str, int] = {}\n        dq = deque(frontier)\n        while dq:\n            cur = dq.popleft()\n            for src, idx in self.rev.get(cur, ()):\n                if src not in dist:\n                    dist[src] = dist[cur] + 1\n                    hop[src] = idx\n                    dq.append(src)\n        self.dist, self.next_hop = dist, hop\n        self.stats["frontier"] = len(frontier)\n\n    def frontier_size(self) -> int:\n        return sum(1 for n in self.nodes.values() if n.open_in(self.active))\n',
    'graft_explore.py': '"""Explorer fallback graft (Pack 4, 2026-08-29) — a model-free frontier walk\ntakes over a stuck level, or the last minutes of a game, and hands back.\n\nInstalled AFTER graft_control. Two triggers, both checked at the top of every\nLLM turn (ToolAgent.analyze) from the session state Pack 2 maintains:\n\n  STALL TIER 3   after the harness RESET tier has fired at least once on this\n                 level and since_new >= TP4_STALL_T3 (default 30) again, and\n                 the explorer has not yet spent its per-level budget:\n                 run frontier_explorer for up to TP4_BUDGET actions (default\n                 800) or until the level changes / game ends. Engine actions\n                 via the gateway cost ~8 ms each, so 800 actions is seconds.\n  ENDGAME        time_remaining <= TP4_ENDGAME_S (default 300) and no level\n                 progress in the last TP4_ENDGAME_QUIET_S (default 900):\n                 run the explorer until TP4_ENDGAME_STOP_S (30) remain.\n\nAfter a run the next user prompt carries a HARNESS NOTE (what was tried, how\nmany actions, whether a level was completed) and Pack 2\'s stall counters are\nreset. Measured offline (bench_explorer.py): alone, the explorer reaches L1 on\n10/25 public games within 800 actions and 15/25 within 3000; on the LLM\'s\nzero-level games it adds 2 (800) to 4 (3000). Its job is unlocking depth for\nthe LLM, not scoring the level itself (an 800-action level scores ~0).\n"""\nfrom __future__ import annotations\n\nimport os\nimport time\nfrom typing import Any\n\n_STATE = {"installed": False}\n_OFF = {"0", "false", "no", "off"}\n_ANALYZE_STOCK: dict[str, Any] = {}\n\nEXPLORER_NOTE = (\n    "HARNESS NOTE: a model-free explorer took over for {n} actions ({why}); it {outcome}. "\n    "It tried {tested} distinct (state, action) pairs across {nodes} board states. "\n    "All of its actions are in `history`. {tail}"\n)\n\n\ndef _env(name: str, default: str) -> str:\n    raw = os.environ.get(name)\n    return default if raw is None or not raw.strip() else raw.strip()\n\n\ndef enabled() -> bool:\n    return _env("TP4_ENABLE", "1").lower() not in _OFF\n\n\ndef _int(name: str, default: int) -> int:\n    try:\n        return int(_env(name, str(default)))\n    except ValueError:\n        return default\n\n\ndef stall_t3() -> int:\n    return max(1, _int("TP4_STALL_T3", 30))\n\n\ndef budget() -> int:\n    return max(1, _int("TP4_BUDGET", 800))\n\n\ndef runs_per_level() -> int:\n    return max(0, _int("TP4_RUNS_PER_LEVEL", 1))\n\n\ndef endgame_s() -> float:\n    return float(_int("TP4_ENDGAME_S", 300))\n\n\ndef endgame_quiet_s() -> float:\n    return float(_int("TP4_ENDGAME_QUIET_S", 900))\n\n\ndef endgame_stop_s() -> float:\n    return float(_int("TP4_ENDGAME_STOP_S", 30))\n\n\ndef status() -> dict[str, Any]:\n    return {"installed": _STATE["installed"], "enabled": enabled(), "stall_t3": stall_t3(),\n            "budget": budget(), "runs_per_level": runs_per_level(), "endgame_s": endgame_s()}\n\n\n# ------------------------------------------------------------- the run ---\ndef run_explorer(session: Any, solver_mod: Any, arcengine: Any, fe: Any, *, max_actions: int,\n                 deadline_s: float | None = None, why: str = "stall") -> dict[str, Any]:\n    """Drive the session\'s engine with the frontier explorer until the level\n    changes, the game ends, the budget is spent, or the deadline passes."""\n    game = session.game\n    level0 = int(solver_mod._level_number(game))\n    ex = fe.FrontierExplorer(seed=int(time.time()) & 0xFFFF)\n    grid = solver_mod._grid_from_state(game.current_state)\n    key = ex.observe(grid, list(game.current_state.available_actions))\n    n = 0\n    outcome = "made no level progress"\n    t0 = time.monotonic()\n    while n < max_actions:\n        if deadline_s is not None and (time.monotonic() - t0) >= deadline_s:\n            outcome = "stopped at the time limit"\n            break\n        try:\n            if session.should_stop():\n                outcome = "stopped (session ending)"\n                break\n        except Exception:  # noqa: BLE001\n            pass\n        if solver_mod._is_engine_game_over(game):\n            action = arcengine.ActionInput(id=arcengine.GameAction.RESET, data={})\n            session._execute_action(action, batch_index=1, batch_size=1, generated_tokens=0)\n            n += 1\n            grid = solver_mod._grid_from_state(game.current_state)\n            key = ex.observe(grid, list(game.current_state.available_actions))\n            continue\n        idx, _why = ex.choose(key)\n        cand = ex.nodes[key].candidates[idx]\n        action = arcengine.ActionInput(id=arcengine.GameAction.from_name(cand.action), data=dict(cand.data))\n        payload = session._execute_action(action, batch_index=1, batch_size=1, generated_tokens=0)\n        n += 1\n        if payload.get("run_complete"):\n            outcome = "COMPLETED THE GAME"\n            break\n        if payload.get("level_completed") or int(solver_mod._level_number(game)) != level0:\n            outcome = f"COMPLETED level {level0}; you are now on level {solver_mod._level_number(game)}"\n            break\n        grid = solver_mod._grid_from_state(game.current_state)\n        new_key = ex.observe(grid, list(game.current_state.available_actions))\n        ex.record(key, idx, new_key)\n        key = new_key\n    tested = ex.stats["edges"] + ex.stats["noops"]\n    return {"actions": n, "outcome": outcome, "tested": tested, "nodes": ex.stats["nodes"],\n            "why": why, "wall_s": round(time.monotonic() - t0, 1)}\n\n\ndef _note(rec: dict[str, Any]) -> str:\n    tail = ("Build on the new level from a fresh look at `current_frame`."\n            if "COMPLETED" in rec["outcome"] else\n            "The explored actions did not progress the level: prefer hypotheses that explain "\n            "why, and try targets or sequences the explorer could not (it never chains actions "\n            "with intent).")\n    return EXPLORER_NOTE.format(n=rec["actions"], why=rec["why"], outcome=rec["outcome"],\n                                tested=rec["tested"], nodes=rec["nodes"], tail=tail)\n\n\n# --------------------------------------------------------------- install ---\ndef install() -> str:\n    if _STATE["installed"]:\n        return "explore: SKIP (already applied)"\n    try:\n        import graft_control as tc  # noqa: PLC0415\n        import frontier_explorer as fe  # noqa: PLC0415\n    except Exception as exc:  # noqa: BLE001\n        return f"explore: SKIP (module missing: {exc!r})"\n    if not tc._STATE.get("installed"):\n        return "explore: SKIP (graft_control not installed)"\n    try:\n        import arcengine  # noqa: PLC0415\n        from inference.agent import tool_agent as agent_mod  # noqa: PLC0415\n        from inference.framework import solver as solver_mod  # noqa: PLC0415\n    except Exception as exc:  # noqa: BLE001\n        return f"explore: SKIP (import failed: {exc!r})"\n    agent_cls = agent_mod.ToolAgent\n    stock_analyze = agent_cls.analyze\n\n    def analyze(self, state_path, action_num, valid_actions=None, step_env=None, **kwargs):\n        try:\n            sess = getattr(step_env, "__self__", None)\n            if sess is not None and enabled():\n                st = tc._state(sess)\n                runs = getattr(st, "explorer_runs_this_level", 0)\n                level = st.level\n                if getattr(st, "explorer_level", None) != level:\n                    st.explorer_level = level\n                    st.explorer_runs_this_level = 0\n                    runs = 0\n                rec = None\n                timing = {}\n                try:\n                    timing = sess.timing_payload()\n                except Exception:  # noqa: BLE001\n                    pass\n                remaining = timing.get("time_remaining_seconds")\n                quiet = time.monotonic() - getattr(st, "last_progress_at", time.monotonic())\n                if (st.since_new >= stall_t3() and getattr(st, "resets_this_level", 0) >= 1\n                        and runs < runs_per_level()):\n                    rec = run_explorer(sess, solver_mod, arcengine, fe, max_actions=budget(),\n                                       deadline_s=None if remaining is None else max(5.0, remaining - endgame_stop_s()),\n                                       why=f"{st.since_new} actions without a new board state")\n                    st.explorer_runs_this_level = runs + 1\n                elif (remaining is not None and remaining <= endgame_s() and quiet >= endgame_quiet_s()\n                      and not getattr(st, "endgame_done", False)):\n                    st.endgame_done = True\n                    rec = run_explorer(sess, solver_mod, arcengine, fe, max_actions=10 ** 6,\n                                       deadline_s=max(1.0, remaining - endgame_stop_s()),\n                                       why="endgame: no level progress recently and the clock is almost out")\n                if rec is not None:\n                    st.since_new = 0\n                    st.streak = 0\n                    st.last_reset_note = _note(rec)\n                    st.explorer_last = rec\n                    try:\n                        sess.write_runtime_state()\n                    except Exception:  # noqa: BLE001\n                        pass\n                    action_num = sess.action_count\n                    valid_actions = solver_mod._engine_action_names(sess.game)\n        except Exception:  # noqa: BLE001\n            pass\n        return _ANALYZE_STOCK["fn"](self, state_path, action_num, valid_actions=valid_actions,\n                                    step_env=step_env, **kwargs)\n\n    _ANALYZE_STOCK["fn"] = stock_analyze\n    analyze._tp4_stock = stock_analyze\n    agent_cls.analyze = analyze\n\n    # progress clock for the endgame trigger: any level change stamps it\n    stock_after = tc._after_action\n\n    def after_action(st, before, after, payload):\n        prev_level = st.level\n        stock_after(st, before, after, payload)\n        if st.level != prev_level or not hasattr(st, "last_progress_at"):\n            st.last_progress_at = time.monotonic()\n\n    after_action._tp4_stock = stock_after\n    tc._after_action = after_action\n\n    _STATE["installed"] = True\n    return "explore: OK"\n',
    'graft_emission.py': '"""Emission graft (Pack 5, 2026-08-30) — attack the two measured modal\nfailures of the stock loop (docs/research-2026-08-29/R7-stock-failure-\nforensics-2026-08-30.md):\n\n  G  analysis-paralysis: 63% of wall time sits in model calls that execute\n     ZERO env actions; 47% of calls end at the 60 s yield without acting\n     (tn36: 50 identical calls, 0 actions in 2.2 h).\n  amnesia-by-channel: in 4/16 games the assistant text is empty all run, so\n     the note harvest gets nothing and the model restarts from scratch every\n     call (re86: 54 responses with 0 content chars). The world model lives in\n     the hidden REASONING channel (Feng\'s 66.8% field finding).\n\nTwo independent, flag-gated behaviours (installed after graft_throughput;\nTP5_ENABLE=0 = pass-through):\n\n  (A) TP5_WM_FROM_REASONING (default 1)\n      When a model response carries NO parsable note in its assistant text,\n      harvest `World model:`-style labelled blocks from the reasoning text\n      instead. Purely additive: the assistant channel wins when non-empty.\n\n  (B) TP5_ACT_FLOOR (default 3)\n      Track consecutive model calls WITHOUT an executed env action, across\n      turns, per agent (the session\'s own counter — reset whenever an action\n      executes). When the streak reaches the floor, the NEXT request forces\n      `tool_choice` to the python function and appends one user line telling\n      the model to act on its best current hypothesis. Mechanism reused from\n      submission/_effort_medium/graft_effort.py (the dead-retry seam), which\n      passed its offline suite; the trigger here is broader (no action\n      executed, not just empty completions) and the injected line names the\n      analysis streak. TP5_ACT_FLOOR=0 disables.\n\nFail-open: errors fall through to stock behaviour.\n"""\nfrom __future__ import annotations\n\nimport os\nimport threading\nfrom typing import Any\n\n_STATE = {"installed": False}\n_RUN_STOCK: dict = {}\n_OFF = {"0", "false", "no", "off"}\n_tls = threading.local()\n\nACT_LINE = (\n    "You have made {n} analyses in a row without executing any action. Analysis is no longer "\n    "buying information the board can\'t give you faster. Call the `python` tool NOW and make it "\n    "end with `action(...)` executing the best action or short batch under your current best "\n    "hypothesis — acting and observing the result IS the experiment."\n)\nFORCED_TOOL_CHOICE = {"type": "function", "function": {"name": "python"}}\n\n\ndef _env(name: str, default: str) -> str:\n    raw = os.environ.get(name)\n    return default if raw is None or not raw.strip() else raw.strip()\n\n\ndef enabled() -> bool:\n    return _env("TP5_ENABLE", "1").lower() not in _OFF\n\n\ndef wm_from_reasoning() -> bool:\n    return enabled() and _env("TP5_WM_FROM_REASONING", "1").lower() not in _OFF\n\n\ndef act_floor() -> int:\n    if not enabled():\n        return 0\n    try:\n        return max(0, int(_env("TP5_ACT_FLOOR", "3")))\n    except ValueError:\n        return 3\n\n\ndef status() -> dict[str, Any]:\n    return {"installed": _STATE["installed"], "enabled": enabled(),\n            "wm_from_reasoning": wm_from_reasoning(), "act_floor": act_floor()}\n\n\ndef _calls_without_action(agent: Any) -> int:\n    return int(getattr(agent, "_tp5_calls_without_action", 0) or 0)\n\n\ndef install() -> str:\n    if _STATE["installed"]:\n        return "emission: SKIP (already applied)"\n    try:\n        from inference.agent import tool_agent as agent_mod\n        from inference.utils import openai_compat as compat_mod\n    except Exception as exc:  # noqa: BLE001\n        return f"emission: SKIP (import failed: {exc!r})"\n    agent_cls = getattr(agent_mod, "ToolAgent", None)\n    if agent_cls is None:\n        return "emission: SKIP (missing ToolAgent)"\n    for name in ("_update_summarized_knowledge_from_assistant", "_chat_completion",\n                 "_run_python_tool", "_extract_scientist_note"):\n        if getattr(agent_cls, name, getattr(agent_mod, name, None)) is None:\n            return f"emission: SKIP ({name} missing)"\n    if getattr(agent_mod, "build_chat_payload", None) is None:\n        return "emission: SKIP (build_chat_payload rebind seam missing)"\n\n    # (A) note harvest falls back to the reasoning channel ------------------\n    stock_chat = agent_cls._chat_completion\n\n    def chat(self, messages, **kwargs):\n        # act-floor: arm the forced tool choice for THIS request when the\n        # analysis streak has reached the floor\n        floor = act_floor()\n        force = floor > 0 and _calls_without_action(self) >= floor\n        _tls.force_now = force\n        _tls.act_line_n = _calls_without_action(self)\n        try:\n            if force:\n                messages = list(messages) + [{"role": "user", "content": ACT_LINE.format(n=_tls.act_line_n)}]\n            result = stock_chat(self, messages, **kwargs)\n        finally:\n            _tls.force_now = False\n        try:\n            self._tp5_calls_without_action = _calls_without_action(self) + 1\n            if wm_from_reasoning():\n                message = getattr(result, "message", None)\n                if isinstance(message, dict):\n                    content = agent_mod._normalize_message_content(message.get("content", ""))\n                    if not agent_mod._extract_scientist_note(str(content or "")):\n                        reasoning = agent_mod._extract_reasoning_text(message)\n                        note = agent_mod._extract_scientist_note(str(reasoning or ""))\n                        if note:\n                            for key, value in note.items():\n                                if value:\n                                    self._summarized_knowledge[key] = value\n        except Exception:  # noqa: BLE001\n            pass\n        return result\n\n    chat._tp5_stock = stock_chat\n    agent_cls._chat_completion = chat\n\n    # forced tool choice rides build_chat_payload only while armed ----------\n    stock_build = compat_mod.build_chat_payload\n\n    def build(*args, **kwargs):\n        try:\n            if getattr(_tls, "force_now", False) and kwargs.get("tools"):\n                kwargs = dict(kwargs)\n                kwargs["tool_choice"] = FORCED_TOOL_CHOICE\n        except Exception:  # noqa: BLE001\n            pass\n        return stock_build(*args, **kwargs)\n\n    build._tp5_stock = stock_build\n    compat_mod.build_chat_payload = build\n    agent_mod.build_chat_payload = build  # dual-namespace rebind (imported by name)\n\n    # streak reset: an executed action clears the counter -------------------\n    _RUN_STOCK["fn"] = agent_cls._run_python_tool\n\n    def run_tool(self, state_path, arguments):\n        result = _RUN_STOCK["fn"](self, state_path, arguments)\n        try:\n            if getattr(result, "step_executed", False):\n                self._tp5_calls_without_action = 0\n        except Exception:  # noqa: BLE001\n            pass\n        return result\n\n    run_tool._tp5_stock = _RUN_STOCK["fn"]\n    agent_cls._run_python_tool = run_tool\n\n    _STATE["installed"] = True\n    return "emission: OK"\n',
    'graft_economy.py': '"""Action-economy graft (TP6, 2026-08-31) — tell the model the truth about\nscoring, and surface its own action spend.\n\nMeasured leak (submission/_flashnext_smoke/results_v4 + scoring math): on the\nFlash-Next smoke, 1.63 local pts/game are forfeited on COMPLETED levels done\nfar over the human baseline (ka59 L1 296 acts vs 28 -> 0.03 pts; bp35 189/21;\nsc25 83/36; quadratic penalty). The stock duck prompt NEVER mentions that\nactions are scored, so the model optimises for progress only.\n\nOne flag-gated behaviour (TP6_ENABLE, default 1): append a short ACTION\nECONOMY block to the user prompt each turn:\n  - actions are scored: per level, score = (human_baseline / your_actions)^2,\n    so finishing a level in 2x the needed actions costs 75% of its points;\n  - the current level\'s action count so far (from actions_per_level via the\n    step summary when available, else the turn-header action counter);\n  - three rules: probe ONCE per hypothesis, never repeat an action that\n    already changed nothing (results carry board_changed), and once the\n    mechanic is verified execute the remaining plan as ONE batch.\n\nPurely prompt-side; no seam behaviour changes. Installed on ToolAgent.\n_build_user_prompt after any other graft (append-only, fail-open).\n"""\nfrom __future__ import annotations\n\nimport os\nfrom typing import Any\n\n_STATE = {"installed": False}\n_OFF = {"0", "false", "no", "off"}\n\nECONOMY_BLOCK = (\n    "ACTION ECONOMY (scoring truth): every level is scored (human_baseline / your_actions)^2 — "\n    "taking twice the needed actions keeps only a quarter of the level\'s points, and points are "\n    "only awarded for COMPLETED levels. You have executed {n} actions on this level so far. "\n    "Probe once per hypothesis; never repeat an action that already changed nothing; once a "\n    "mechanic is verified, execute the whole remaining plan as one action([...]) batch."\n)\n\n\ndef enabled() -> bool:\n    raw = os.environ.get("TP6_ENABLE")\n    value = "1" if raw is None or not raw.strip() else raw.strip()\n    return value.lower() not in _OFF\n\n\ndef status() -> dict[str, Any]:\n    return {"installed": _STATE["installed"], "enabled": enabled()}\n\n\ndef _level_actions(agent: Any, action_num: int) -> int:\n    try:\n        summary = getattr(agent, "_last_step_summary", None) or {}\n        end = summary.get("end_action_num")\n        # best available proxy: total actions this game; per-level split is not\n        # visible to the agent, so state the game counter when level unknown\n        return int(end if end is not None else max(0, action_num))\n    except Exception:  # noqa: BLE001\n        return max(0, int(action_num or 0))\n\n\ndef install() -> str:\n    if _STATE["installed"]:\n        return "economy: SKIP (already applied)"\n    try:\n        from inference.agent import tool_agent as agent_mod\n    except Exception as exc:  # noqa: BLE001\n        return f"economy: SKIP (import failed: {exc!r})"\n    cls = getattr(agent_mod, "ToolAgent", None)\n    if cls is None or getattr(cls, "_build_user_prompt", None) is None:\n        return "economy: SKIP (seam missing)"\n\n    stock = cls._build_user_prompt\n\n    def build(self, action_num, **kwargs):\n        text = stock(self, action_num, **kwargs)\n        if not enabled():\n            return text\n        try:\n            return text + "\\n" + ECONOMY_BLOCK.format(n=_level_actions(self, action_num))\n        except Exception:  # noqa: BLE001\n            return text\n\n    build._tp6_stock = stock\n    cls._build_user_prompt = build\n    _STATE["installed"] = True\n    return "economy: OK"\n',
}

_G_DIR = WORKING_DIR / "tuned_bundle"
_G_DIR.mkdir(parents=True, exist_ok=True)
for _name, _src in _GRAFT_SOURCES.items():
    (_G_DIR / _name).write_text(_src, encoding="utf-8")
if str(_G_DIR) not in sys.path:
    sys.path.insert(0, str(_G_DIR))
import importlib as _il
_tp = _il.import_module("graft_throughput"); _st = _tp.install()
assert _st == "throughput: OK", _st
_tc = _il.import_module("graft_control"); _st = _tc.install()
assert _st == "control: OK", _st
_te = _il.import_module("graft_explore"); _st = _te.install()
assert _st == "explore: OK", _st
_tm = _il.import_module("graft_emission"); _st = _tm.install()
assert _st == "emission: OK", _st
_t6 = _il.import_module("graft_economy"); _st = _t6.install()
assert _st == "economy: OK", _st
print("[tuned] all grafts installed; enabled:",
      {m.__name__.split("_")[1]: m.enabled() for m in (_tp, _tc, _te, _tm, _t6)})


In [ ]:
# ---- flashnext smoke telemetry: THE READ. Per-game actions / levels / score /
# turns / tokens from the framework mirror + the ToolAgent session counters,
# vLLM /metrics deltas (prompt + generation tokens, prefix-cache queries/hits,
# preemptions), and a 60 s queue sampler: 28 client workers vs their
# --max-num-seqs 22 means requests queue by design — running/waiting/kv are
# part of the record.
import json as _tel_json
import re as _tel_re
import threading as _tel_threading
import time as _tel_time
import urllib.request as _tel_rq

from inference.framework import solver as _solver_mod

_tel_lock = _tel_threading.Lock()
_TEL_PATH = WORKING_DIR / "flashnext_smoke_results.json"
FN_SESSIONS = {}      # phase -> {game_id: {...}} filled at session exit
FN_PHASES = []        # ordered phase records
_FN_CURRENT = {"phase": None, "t0": None, "metrics0": None,
               "sampler_stop": None, "queue_samples": None}


def _metrics_url():
    base = (os.environ.get("LOCAL_ANALYZER_BASE_URL") or "http://127.0.0.1:1234/v1").rstrip("/")
    if base.endswith("/v1"):
        base = base[:-3]
    return base + "/metrics"


_COUNTER_KEYS = (
    "vllm:prompt_tokens_total", "vllm:generation_tokens_total",
    "vllm:prefix_cache_queries_total", "vllm:prefix_cache_hits_total",
    "vllm:request_success_total", "vllm:num_preemptions_total",
)
_GAUGE_KEYS = (
    "vllm:num_requests_running", "vllm:num_requests_waiting",
    "vllm:kv_cache_usage_perc",
)


def _fn_scrape(keys):
    out = {}
    try:
        with _tel_rq.urlopen(_metrics_url(), timeout=20) as resp:
            text = resp.read().decode("utf-8", errors="replace")
    except Exception as exc:  # noqa: BLE001
        return {"error": repr(exc)}
    for line in text.splitlines():
        if not line or line.startswith("#"):
            continue
        for key in keys:
            # exact metric name only ("...waiting" must not match "...waiting_by_reason")
            if line.startswith(key + "{") or line.startswith(key + " "):
                m = _tel_re.match(r"^\S+(?:\{[^}]*\})?\s+([0-9.eE+-]+)", line)
                if m:
                    try:
                        out[key] = out.get(key, 0.0) + float(m.group(1))
                    except ValueError:
                        pass
    return out


def _fn_phase_begin(phase):
    stop = _tel_threading.Event()
    samples = []

    def _queue_sampler():
        while not stop.is_set():
            g = _fn_scrape(_GAUGE_KEYS)
            if "error" not in g:
                g["t"] = round(_tel_time.time(), 1)
                samples.append(g)
            stop.wait(60)

    with _tel_lock:
        _FN_CURRENT.update({"phase": phase, "t0": _tel_time.monotonic(),
                            "metrics0": _fn_scrape(_COUNTER_KEYS),
                            "sampler_stop": stop, "queue_samples": samples})
        FN_SESSIONS.setdefault(phase, {})
    _tel_threading.Thread(target=_queue_sampler, daemon=True).start()
    print(f"[fn-tel] phase {phase} begin metrics0={_FN_CURRENT['metrics0']}", flush=True)


def _fn_phase_end(phase, game_runs):
    stop = _FN_CURRENT.get("sampler_stop")
    if stop is not None:
        stop.set()
    m1 = _fn_scrape(_COUNTER_KEYS)
    m0 = _FN_CURRENT.get("metrics0") or {}
    delta = {k: (m1.get(k, 0.0) - m0.get(k, 0.0)) for k in _COUNTER_KEYS if k in m1}
    wall = _tel_time.monotonic() - (_FN_CURRENT.get("t0") or _tel_time.monotonic())
    games = []
    for game_run in game_runs:
        apl = list(game_run.actions_per_level or [])
        actions = sum(apl) if apl else len(game_run.history)
        sess = FN_SESSIONS.get(phase, {}).get(game_run.game_id, {})
        games.append({
            "game_id": game_run.game_id,
            "state": game_run.state,
            "levels_completed": game_run.levels_completed,
            "number_of_levels": game_run.number_of_levels,
            "final_score": game_run.final_score,
            "actions": actions,
            "actions_per_level": apl,
            "base_actions_per_level": list(game_run.base_actions_per_level or []),
            "wallclock_s": game_run.final_wallclock_seconds,
            "solver_note": game_run.solver_note,
            "turns": sess.get("turns"),
            "session_total_tokens": sess.get("total_tokens"),
            "session_generated_tokens": sess.get("generated_tokens"),
            "history_messages_at_exit": sess.get("history_messages"),
            "context_budget_tokens": sess.get("context_budget_tokens"),
            "yield_seconds": sess.get("yield_seconds"),
            "tool_steps": sess.get("tool_steps"),
        })
    n = max(1, len(games))
    queries = delta.get("vllm:prefix_cache_queries_total", 0.0)
    hits = delta.get("vllm:prefix_cache_hits_total", 0.0)
    gen = delta.get("vllm:generation_tokens_total", 0.0)
    prompt = delta.get("vllm:prompt_tokens_total", 0.0)
    q = list(_FN_CURRENT.get("queue_samples") or [])
    running = [s["vllm:num_requests_running"] for s in q if "vllm:num_requests_running" in s]
    waiting = [s["vllm:num_requests_waiting"] for s in q if "vllm:num_requests_waiting" in s]
    kv = [s["vllm:kv_cache_usage_perc"] for s in q if "vllm:kv_cache_usage_perc" in s]
    rec = {
        "phase": phase,
        "harness": "stock-duck",
        "server_tag": CURRENT_SERVER["tag"],
        "analyzer_env": {k: v for k, v in os.environ.items()
                         if k.startswith(("LOCAL_ANALYZER", "INFERENCE_ANALYZER", "MULTIMODAL"))},
        "wall_s": round(wall, 1),
        "games": games,
        "n_games": len(games),
        "mean_actions": round(sum(g["actions"] for g in games) / n, 2),
        "mean_levels": round(sum(g["levels_completed"] for g in games) / n, 3),
        "mean_score": round(sum(float(g["final_score"] or 0.0) for g in games) / n, 4),
        "zero_level_games": sum(1 for g in games if g["levels_completed"] == 0),
        "mean_turns": round(sum(float(g["turns"] or 0) for g in games) / n, 1),
        "metrics_delta": delta,
        "prefix_hit_rate": round(hits / queries, 4) if queries else None,
        "prefill_per_gen_token": round(prompt / gen, 2) if gen else None,
        "gen_tok_s": round(gen / wall, 1) if wall else None,
        "queue": {
            "n_samples": len(q),
            "running_mean": round(sum(running) / len(running), 1) if running else None,
            "running_max": max(running) if running else None,
            "waiting_mean": round(sum(waiting) / len(waiting), 1) if waiting else None,
            "waiting_max": max(waiting) if waiting else None,
            "kv_usage_mean": round(sum(kv) / len(kv), 3) if kv else None,
            "kv_usage_max": round(max(kv), 3) if kv else None,
        },
    }
    with _tel_lock:
        FN_PHASES.append(rec)
    try:
        _TEL_PATH.write_text(_tel_json.dumps(
            {"phases": FN_PHASES, "phase_errors": FN_PHASE_ERRORS}, indent=1, default=str),
            encoding="utf-8")
    except Exception:  # noqa: BLE001
        pass
    print(f"[fn-tel] phase {phase} end: games={rec['n_games']} mean_actions={rec['mean_actions']} "
          f"mean_levels={rec['mean_levels']} mean_score={rec['mean_score']} "
          f"zero_level={rec['zero_level_games']} turns={rec['mean_turns']} "
          f"prefix_hit={rec['prefix_hit_rate']} prefill/gen={rec['prefill_per_gen_token']} "
          f"gen_tok_s={rec['gen_tok_s']} queue={rec['queue']} wall={rec['wall_s']}s", flush=True)


# Session exit seam: record turns + token counters per game for the phase.
_inner_play = _solver_mod._HarnessGameSession.play


def _tel_play(self):
    try:
        return _inner_play(self)
    finally:
        try:
            gid = getattr(getattr(self.game, "game_run", None), "game_id", "?")
            an = self.analyzer
            rec = {
                "turns": int(getattr(self, "analysis_step", 0) or 0),
                "total_tokens": int(getattr(an, "_session_total_tokens", 0) or 0),
                "generated_tokens": int(getattr(an, "_session_generated_tokens", 0) or 0),
                "history_messages": len(getattr(an, "_history_messages", []) or []),
                "context_budget_tokens": getattr(an, "_context_budget_tokens", None),
                "yield_seconds": getattr(an, "_yield_seconds", None),
                "tool_steps": getattr(an, "_tool_steps", None),
            }
            with _tel_lock:
                FN_SESSIONS.setdefault(_FN_CURRENT.get("phase") or "?", {})[gid] = rec
        except Exception:  # noqa: BLE001
            pass


if not getattr(_solver_mod._HarnessGameSession.play, "_fn_tel", False):
    _tel_play._fn_tel = True
    _solver_mod._HarnessGameSession.play = _tel_play

print("[fn-tel] installed; metrics url =", _metrics_url(), flush=True)


In [ ]:
run_context = contextlib.nullcontext() if run_as_submission else _tee_to_file(WORKING_DIR / "stdout.log")
with run_context:
    preamble = (BUNDLE_DIR / "preamble.txt").read_text(encoding="utf-8")
    print(preamble)
    print(f"deploy.kaggle: working_dir             = {WORKING_DIR}")
    print(f"deploy.kaggle: run_as_submission       = {run_as_submission}")
    print(f"deploy.kaggle: competition_rerun       = {true_submission}")
    print(f"deploy.kaggle: soft_end_time           = {soft_end}")
    print("---")

    bundled_git_status = BUNDLE_DIR / "git_status.txt"
    if bundled_git_status.is_file():
        (WORKING_DIR / "git_status.txt").write_text(
            bundled_git_status.read_text(encoding="utf-8"),
            encoding="utf-8",
        )

    if true_submission:
        # Competition reruns use Kaggle's live gateway instead of the bundled offline games.
        os.environ.setdefault("ARC_API_KEY", "test-key-123")
        os.environ.setdefault("ARC_BASE_URL", "http://gateway:8001/")
        os.environ.setdefault("SCHEME", "http")
        os.environ.setdefault("HOST", "gateway")
        os.environ.setdefault("PORT", "8001")
        os.environ.setdefault("OPERATION_MODE", "competition")
        os.environ.setdefault("ENVIRONMENTS_DIR", "")
        os.environ.setdefault("RECORDINGS_DIR", str(WORKING_DIR / "server_recording"))

        deadline = time.monotonic() + 600.0
        last_error = ""
        while time.monotonic() < deadline:
            try:
                with urlopen("http://gateway:8001/api/games", timeout=10) as response:
                    if response.status < 500:
                        break
            except Exception as exc:
                last_error = repr(exc)
            time.sleep(5)
        else:
            raise RuntimeError(f"Kaggle gateway did not become ready: {last_error}")

        bm.games = _competition_games()
        bm.n_passes = 1
        bm.game_weights = None

    try:
        # flashnext smoke: ONE phase — the STOCK duck against the Flash-Next
        # server. The scored-rerun path takes exactly one pass with the
        # competition games (and would need the 27B kernel, not this one).
        for _phase_name, _phase_games, _phase_cap, _phase_env in SMOKE_PHASES:
            for _k, _v in _phase_env.items():
                os.environ[_k] = _v
            if not run_as_submission:
                FN_ALL_RUNS.extend(bm.game_runs)
                bm.game_runs = []
                bm.games = [GameAPI(env_name=_n, arcade_spec=_spec) for _n in _phase_games]
                bm.n_passes = 1
                bm.game_weights = None
                bm.label = "flashnext-smoke-" + _phase_name
                bm.solver.max_runtime_s_per_game = float(_phase_cap)
                print(f"=== PHASE {_phase_name}: {len(_phase_games)} games env={_phase_env} "
                      f"per_game_cap={_phase_cap}s concurrency={bm.solver.concurrency} "
                      f"model={os.environ.get('INFERENCE_ANALYZER_MODEL')} ===", flush=True)
            _fn_phase_begin(_phase_name)
            try:
                await bm.run(
                    soft_end_time=soft_end,
                    runtime_environment=target,
                    minimal_diagnostics=run_as_submission,
                )
            except Exception as _phase_exc:  # noqa: BLE001
                import traceback

                FN_PHASE_ERRORS.append(f"{_phase_name}: {type(_phase_exc).__name__}: {_phase_exc}")
                print(f"PHASE {_phase_name} RAISED {type(_phase_exc).__name__}: {_phase_exc}",
                      flush=True)
                traceback.print_exc()
            try:
                _fn_phase_end(_phase_name, list(bm.game_runs))
            except Exception:  # noqa: BLE001
                pass
            if run_as_submission:
                break
        if not true_submission and Path("/kaggle/input").exists():
            try:
                import pandas as pd

                submission = pd.DataFrame(
                    data=[["1_0", "1", True, 1]],
                    columns=["row_id", "game_id", "end_of_game", "score"],
                )
                submission.to_parquet(WORKING_DIR / "submission.parquet", index=False)
            except Exception as exc:
                print(f"taaf.kaggle: could not write offline dummy submission: {exc!r}", flush=True)
    finally:
        _run_shell_commands("teardown_commands.json", label="teardown", check=False)

In [ ]:
# ---- flashnext smoke final report (grep for FLASHNEXT SMOKE / READ) ----
BASELINE_27B_STOCK = {
    "source": "pooled stock phases of arc3-tp-smoke / arc3-tp1b-smoke / arc3-tp1c-smoke "
              "(27B-FP8, same GPU class, same geometry, 2026-08-29)",
    "levels_per_game": "1.00 / 1.04 / 0.84-0.88",
    "zero_level_games": "8-9 of 25",
    "mean_local_score": "3.5-4.9",
}
print("=" * 78)
print("FLASHNEXT SMOKE RESULTS — STOCK duck x Flash-Next NVFP4 (model-swap read)")
print("boot:", RESULTS["verdicts"].get("boot"))
print("27B stock baseline:", BASELINE_27B_STOCK)
by = {p["phase"]: p for p in FN_PHASES}
p = by.get("flashnext")
verdict = "UNREADABLE"
detail = ""
if not p or not p["n_games"]:
    print("PHASE flashnext: MISSING")
else:
    print(f"PHASE flashnext: games={p['n_games']} mean_actions={p['mean_actions']} "
          f"mean_levels={p['mean_levels']} mean_score={p['mean_score']} "
          f"zero_level={p['zero_level_games']} mean_turns={p['mean_turns']} "
          f"prefix_hit={p['prefix_hit_rate']} prefill/gen={p['prefill_per_gen_token']} "
          f"gen_tok_s={p['gen_tok_s']} wall={p['wall_s']}s")
    print(f"queue: {p['queue']}")
    for g in sorted(p["games"], key=lambda g: g["game_id"]):
        print(f"  {g['game_id']}: levels={g['levels_completed']}/{g['number_of_levels']} "
              f"score={g['final_score']} actions={g['actions']} turns={g['turns']} "
              f"gen_tok={g['session_generated_tokens']} state={g['state']}")
    lev = p["mean_levels"]
    zero = p["zero_level_games"]
    detail = (f"levels {lev:.3f} (27B stock 1.00/1.04/0.84-0.88) zero_level {zero}/25 "
              f"(27B 8-9/25) actions {p['mean_actions']} score {p['mean_score']}")
    # Pre-registered rules (README.md, fixed before flight):
    if lev >= 1.3 or (zero <= 6 and lev >= 1.0):
        verdict = "PASS"
    elif lev < 0.9:
        verdict = "FAIL"
    else:
        verdict = "INCONCLUSIVE"
print(f"FLASHNEXT SMOKE READ: {verdict} ({detail}) phase_errors={FN_PHASE_ERRORS}")
results = {
    "kernel": "arc3-flashnext-smoke",
    "harness": "stock-duck",
    "boot": RESULTS["verdicts"].get("boot"),
    "baseline_27b_stock": BASELINE_27B_STOCK,
    "phases": FN_PHASES,
    "phase_errors": FN_PHASE_ERRORS,
    "verdict": verdict,
    "detail": detail,
}
(WORKING_DIR / "flashnext_smoke_results.json").write_text(
    json.dumps(results, indent=1, default=str), encoding="utf-8")
print("wrote", WORKING_DIR / "flashnext_smoke_results.json")
